## Testing the models on Monk Skin Tone Dataset

In [7]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import stone

In [8]:
def to_bin(x):
    if 1 <= x <= 3:
        return 0
    elif 4 <= x <= 7:
        return 1
    else:
        return 2

def accuracy_exact(y_true, y_pred):
    return (y_true == y_pred).mean()

def accuracy_pm1(y_true, y_pred):
    return (abs(y_true - y_pred) <= 1).mean()

def accuracy_bins(y_true, y_pred):
    y_true_bin = np.array([to_bin(x) for x in y_true])
    y_pred_bin = np.array([to_bin(x) for x in y_pred])
    return (y_true_bin == y_pred_bin).mean()

## SkinToneClassifier Library

Link: https://pypi.org/project/skin-tone-classifier/

In [9]:
def evaluate_dataset(csv_path, image_root, file_path_column="image_path",
                     label_column="mst_label", subject_id_column=None,
                     output_csv="results.csv"):

    df = pd.read_csv(csv_path)

    monk_hex_palette = [
    "#f6ede4","#f3e7db","#f7ead0","#eadaba","#d7bd96",
    "#a07e56","#825c43","#604134","#3a312a","#292420"
    ]
    
    monk_labels = ["1","2","3","4","5","6","7","8","9","10"]

    results = []  # to store per-image predictions for CSV output

    for _, row in tqdm(df.iterrows(), total=len(df)):

        # Build image path
        img_path = (
            os.path.join(image_root, row[file_path_column])
            if subject_id_column is None
            else os.path.join(image_root, row[subject_id_column], row[file_path_column])
        )

        if not os.path.exists(img_path):
            # print(f"Image not found: {img_path}")
            continue

        # Run estimator
        result = stone.process(
            img_path,
            image_type="color",
            tone_palette=monk_hex_palette,
            tone_labels=monk_labels,
            return_report_image=True,
            min_nbrs=5,
            n_dominant_colors=5
        )

        try:
            face_id = result['faces'][0]['face_id']
            pred_label = int(result['faces'][0]['tone_label'])
        except:
            continue

        true_label = int(row[label_column])
        abs_err = abs(true_label - pred_label)

        results.append({
            "subject_id": row.get(subject_id_column, None),
            "image_path": img_path,
            "true_label": true_label,
            "pred_label": pred_label,
            "abs_error": abs_err,
            "match_exact": 1 if true_label == pred_label else 0,
            "match_pm1": 1 if abs_err <= 1 else 0,
            "true_bin": to_bin(true_label),
            "pred_bin": to_bin(pred_label),
            "match_bin": 1 if to_bin(true_label) == to_bin(pred_label) else 0
        })

    # Convert to DataFrame
    res_df = pd.DataFrame(results)

    # Save per-image output
    res_df.to_csv(output_csv, index=False)
    print(f"Saved detailed predictions to: {output_csv}")

    # Global metrics
    y_true = res_df["true_label"].values
    y_pred = res_df["pred_label"].values

    metrics = {
        "accuracy_exact": accuracy_exact(y_true, y_pred),
        "accuracy_pm1": accuracy_pm1(y_true, y_pred),
        "accuracy_3bins": accuracy_bins(y_true, y_pred)
    }

    # Per-tone metrics
    per_tone = {}
    for tone in range(1, 11):
        df_tone = res_df[res_df["true_label"] == tone]
        if len(df_tone) == 0:
            continue

        per_tone[tone] = {
            "count": len(df_tone),
            "exact": df_tone["match_exact"].mean(),
            "pm1": df_tone["match_pm1"].mean(),
            "bin": df_tone["match_bin"].mean()
        }

    return metrics, per_tone, res_df

### Monk Skin Tone Dataset

In [ ]:
# ---------------------------------------------------------
# RUN EVALUATION
# ---------------------------------------------------------
monk_csv_path = r"G:\Thesis\MonkSkinTone_Dataset\mst-e_data\mst-e_image_details.csv"
monk_image_root = r"G:\Thesis\MonkSkinTone_Dataset\mst-e_data"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\mst-e_data\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv"

metrics, per_tone, results_df = evaluate_dataset(
    monk_csv_path,
    monk_image_root,
    subject_id_column="subject_name",
    file_path_column="image_ID",
    label_column="MST",
    output_csv=output_csv
)

print("=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
          f"Exact={stats['exact']:.3f},  "
          f"±1={stats['pm1']:.3f},  "
          f"Bin={stats['bin']:.3f}")


 16%|█▌        | 244/1546 [03:25<17:22,  1.25it/s]Carlos video 2.mp4 is not found or is not a valid image.
Carlos video 3.mp4 is not found or is not a valid image.
 19%|█▉        | 298/1546 [04:13<17:19,  1.20it/s]Carlos Video.TS.mp4 is not found or is not a valid image.
PXL_20220922_160630474.TS.mp4 is not found or is not a valid image.
 27%|██▋       | 423/1546 [06:11<18:33,  1.01it/s]PXL_20220922_140855069.TS.mp4 is not found or is not a valid image.
PXL_20220922_140731682.TS.mp4 is not found or is not a valid image.
 46%|████▌     | 706/1546 [10:26<12:08,  1.15it/s]PXL_20220922_195601107.TS.mp4 is not found or is not a valid image.
PXL_20220922_195528873.TS.mp4 is not found or is not a valid image.
 52%|█████▏    | 807/1546 [11:54<10:33,  1.17it/s]PXL_20220922_134731392.TS.mp4 is not found or is not a valid image.
PXL_20220922_134651916.TS.mp4 is not found or is not a valid image.
 68%|██████▊   | 1052/1546 [15:23<06:58,  1.18it/s]PXL_20220922_142032122.TS.mp4 is not found or is no

Saved detailed predictions to: G:\Thesis\MonkSkinTone_Dataset\mst-e_data\skin_tone_classifier_library_predictions.csv
=========== GLOBAL RESULTS ===========
Exact accuracy:        0.1115
±1 tolerance accuracy: 0.2814
3-bin accuracy:        0.4251

=========== PER-TONE RESULTS ===========
MST 1:  N=179,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 2:  N=218,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 3:  N=90,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 4:  N=187,  Exact=0.000,  ±1=0.000,  Bin=0.701
MST 5:  N=179,  Exact=0.000,  ±1=0.145,  Bin=0.782
MST 6:  N=165,  Exact=0.170,  ±1=0.764,  Bin=0.764
MST 7:  N=76,  Exact=0.632,  ±1=1.000,  Bin=0.803
MST 8:  N=142,  Exact=0.627,  ±1=0.852,  Bin=0.662
MST 9:  N=142,  Exact=0.000,  ±1=0.401,  Bin=0.401
MST 10:  N=111,  Exact=0.009,  ±1=0.117,  Bin=0.216


In [ ]:
# ---------------------------------------------------------
# RUN EVALUATION
# ---------------------------------------------------------
monk_csv_path = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
monk_image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv"

metrics, per_tone, results_df = evaluate_dataset(
    monk_csv_path,
    monk_image_root,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv
)

print("=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
          f"Exact={stats['exact']:.3f},  "
          f"±1={stats['pm1']:.3f},  "
          f"Bin={stats['bin']:.3f}")


100%|██████████| 1388/1388 [03:36<00:00,  6.41it/s]

Saved detailed predictions to: G:\Thesis\MonkSkinTone_Dataset\skin_tone_classifier_library_predictions.csv
=========== GLOBAL RESULTS ===========
Exact accuracy:        0.1110
±1 tolerance accuracy: 0.2961
3-bin accuracy:        0.4741

=========== PER-TONE RESULTS ===========
MST 1:  N=174,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 2:  N=198,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 3:  N=86,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 4:  N=180,  Exact=0.000,  ±1=0.000,  Bin=0.639
MST 5:  N=167,  Exact=0.000,  ±1=0.132,  Bin=0.784
MST 6:  N=154,  Exact=0.071,  ±1=0.591,  Bin=0.591
MST 7:  N=65,  Exact=0.600,  ±1=1.000,  Bin=0.600
MST 8:  N=130,  Exact=0.692,  ±1=0.908,  Bin=0.777
MST 9:  N=124,  Exact=0.000,  ±1=0.790,  Bin=0.790
MST 10:  N=110,  Exact=0.127,  ±1=0.155,  Bin=0.755


### Casual Conversation v2

In [ ]:
# ---------------------------------------------------------
# RUN EVALUATION
# ---------------------------------------------------------
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv"

metrics, per_tone, results_df = evaluate_dataset(
    csv_path,
    image_root,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv
)

print("=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
          f"Exact={stats['exact']:.3f},  "
          f"±1={stats['pm1']:.3f},  "
          f"Bin={stats['bin']:.3f}")


 25%|██▍       | 45139/184201 [1:26:50<4:20:23,  8.90it/s] 

### FACET

In [14]:
# ---------------------------------------------------------
# RUN EVALUATION
# ---------------------------------------------------------
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv"

metrics, per_tone, results_df = evaluate_dataset(
    csv_path,
    image_root,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv
)

print("=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
          f"Exact={stats['exact']:.3f},  "
          f"±1={stats['pm1']:.3f},  "
          f"Bin={stats['bin']:.3f}")


100%|██████████| 2677/2677 [04:30<00:00,  9.91it/s]


Saved detailed predictions to: G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv
=========== GLOBAL RESULTS ===========
Exact accuracy:        0.0732
±1 tolerance accuracy: 0.2394
3-bin accuracy:        0.4206

=========== PER-TONE RESULTS ===========
MST 1:  N=70,  Exact=0.014,  ±1=0.029,  Bin=0.029
MST 2:  N=559,  Exact=0.011,  ±1=0.016,  Bin=0.016
MST 3:  N=693,  Exact=0.000,  ±1=0.007,  Bin=0.007
MST 4:  N=485,  Exact=0.000,  ±1=0.043,  Bin=0.885
MST 5:  N=349,  Exact=0.077,  ±1=0.490,  Bin=0.888
MST 6:  N=288,  Exact=0.222,  ±1=0.788,  Bin=0.788
MST 7:  N=120,  Exact=0.517,  ±1=0.975,  Bin=0.642
MST 8:  N=67,  Exact=0.522,  ±1=0.910,  Bin=0.522
MST 9:  N=41,  Exact=0.000,  ±1=0.659,  Bin=0.659
MST 10:  N=5,  Exact=0.200,  ±1=0.200,  Bin=1.000


## RandomForest Approach

- Paper - Enhancing Fairness in Machine Learning: Skin Tone Classification Using the Monk Skin Tone Scale

Link: https://www.researchgate.net/publication/386403126_Enhancing_Fairness_in_Machine_Learning_Skin_Tone_Classification_Using_the_Monk_Skin_Tone_Scale

In [16]:
import os
import joblib
import pandas as pd
import numpy as np
from tqdm import tqdm
import cv2
import json

# ---------------------------------------------------------
# FEATURE EXTRACTOR (MUST MATCH TRAINING EXACTLY)
# ---------------------------------------------------------
def extract_hist_features(image_bgr, bins=256):
    img_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    img_ycc = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2YCrCb)
    img_hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    img_lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2Lab)

    hists = []

    for i in range(3):
        h = cv2.calcHist([img_rgb], [i], None, [bins], [0, 256]).flatten()
        hists.append(h)

    hists.append(cv2.calcHist([img_ycc], [0], None, [bins], [0, 256]).flatten())
    hists.append(cv2.calcHist([img_hsv], [2], None, [bins], [0, 256]).flatten())
    hists.append(cv2.calcHist([img_lab], [0], None, [bins], [0, 256]).flatten())

    feat = np.concatenate(hists).astype(np.float32)
    feat /= (feat.sum() + 1e-8)

    return feat

# ---------------------------------------------------------
# MAIN EVALUATION LOGIC WITH SPLIT SUPPORT
# ---------------------------------------------------------
def evaluate_rf_model(
    model_path,
    csv_path,
    image_root,
    file_path_column="image_ID",
    label_column="MST",
    subject_id_column=None,
    person_id_column=None,     
    bins=256,
    split_json=None,           
    split_key=None,           
    output_csv="rf_mst_predictions.csv"
):

    print(f"[INFO] Loading RF model from {model_path}")
    model = joblib.load(model_path)

    df = pd.read_csv(csv_path)

    # -----------------------------------------------------
    # APPLY TRAIN / VAL SPLIT FILTER
    # -----------------------------------------------------
    if split_json is not None:
        print(f"[INFO] Loading split from: {split_json}")

        if person_id_column is None:
            raise ValueError("person_id_column must be provided when using split_json")

        with open(split_json, "r") as f:
            split_data = json.load(f)

        if split_key not in split_data:
            raise ValueError(f"Invalid split_key: {split_key}. Available: {list(split_data.keys())}")

        try:
            person_ids = [int(x) for x in split_data[split_key]]
            df_ids = df[person_id_column].astype(int)
        except Exception:
            person_ids = [str(x) for x in split_data[split_key]]
            df_ids = df[person_id_column].astype(str)

        before = len(df)
        df = df[df_ids.isin(person_ids)].reset_index(drop=True)
        print(f"[INFO] Filtered from {before} → {len(df)} images using split '{split_key}'")
    else:
        print(f"[INFO] No split JSON provided, using all {len(df)} images")

    # -----------------------------------------------------
    # MODEL TYPE
    # -----------------------------------------------------
    is_classifier = hasattr(model, "predict_proba")
    print(f"[INFO] Detected model type: {'Classifier' if is_classifier else 'Regressor'}")

    results = []

    # -----------------------------------------------------
    # MAIN EVALUATION LOOP
    # -----------------------------------------------------
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating RF"):

        img_path = (
            os.path.join(image_root, row[file_path_column])
            if subject_id_column is None
            else os.path.join(image_root, row[subject_id_column], row[file_path_column])
        )

        if not os.path.exists(img_path):
            print(f"[WARN] Image not found: {img_path}")
            continue

        img = cv2.imread(img_path)
        if img is None:
            print(f"[WARN] Failed to load: {img_path}")
            continue

        feat = extract_hist_features(img, bins=bins).reshape(1, -1)

        if is_classifier:
            pred_label = int(model.predict(feat)[0])
        else:
            pred_float = float(model.predict(feat)[0])
            pred_label = int(np.clip(round(pred_float), 1, 10))

        true_label = int(row[label_column])
        abs_err = abs(true_label - pred_label)

        results.append({
            "person_id": row.get(person_id_column, None),
            "image_path": img_path,
            "true_label": true_label,
            "pred_label": pred_label,
            "abs_error": abs_err,
            "match_exact": 1 if true_label == pred_label else 0,
            "match_pm1": 1 if abs_err <= 1 else 0,
            "true_bin": to_bin(true_label),
            "pred_bin": to_bin(pred_label),
            "match_bin": 1 if to_bin(true_label) == to_bin(pred_label) else 0
        })

    # -----------------------------------------------------
    # SAVE CSV
    # -----------------------------------------------------
    res_df = pd.DataFrame(results)
    res_df.to_csv(output_csv, index=False)
    print(f"[INFO] Saved predictions to {output_csv}")

    # -----------------------------------------------------
    # GLOBAL METRICS
    # -----------------------------------------------------
    y_true = res_df["true_label"].values
    y_pred = res_df["pred_label"].values

    metrics = {
        "accuracy_exact": accuracy_exact(y_true, y_pred),
        "accuracy_pm1": accuracy_pm1(y_true, y_pred),
        "accuracy_3bins": accuracy_bins(y_true, y_pred)
    }

    # -----------------------------------------------------
    # PER-TONE METRICS
    # -----------------------------------------------------
    per_tone = {}
    for tone in range(1, 11):
        subset = res_df[res_df["true_label"] == tone]
        if len(subset) == 0:
            continue

        per_tone[tone] = {
            "count": len(subset),
            "exact": subset["match_exact"].mean(),
            "pm1": subset["match_pm1"].mean(),
            "bin": subset["match_bin"].mean()
        }

    return metrics, per_tone, res_df


### Monk Skin Tone Dataset

In [ ]:
# Testing on Monk Skin Tone Dataset
model_path = r"G:\Thesis\Models\RandomForest\Model 2\rf_reg_bins256_RMSE1.323_ACC050.346_20251129_215235.joblib"
monk_csv_path = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
monk_image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\rf_mst_predictions.csv"

metrics, per_tone, df = evaluate_rf_model(
    model_path=model_path,
    csv_path=monk_csv_path,
    image_root=monk_image_root,
    file_path_column="filename",
    label_column="mst_label",
    bins=256,
    output_csv=output_csv
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
            f"Exact={stats['exact']:.3f},  "
            f"±1={stats['pm1']:.3f},  "
            f"Bin={stats['bin']:.3f}")


[INFO] Loading RF model from G:\Thesis\Models\RandomForest\Model 2\rf_reg_bins256_RMSE1.323_ACC050.346_20251129_215235.joblib
[INFO] No split JSON provided, using all 1388 images
[INFO] Detected model type: Regressor


Evaluating RF: 100%|██████████| 1388/1388 [02:18<00:00,  9.99it/s]

[INFO] Saved predictions to G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\rf_mst_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.1318
±1 tolerance accuracy: 0.3350
3-bin accuracy:        0.4099

=========== PER-TONE RESULTS ===========
MST 1:  N=174,  Exact=0.000,  ±1=0.000,  Bin=0.017
MST 2:  N=198,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 3:  N=86,  Exact=0.000,  ±1=0.186,  Bin=0.000
MST 4:  N=180,  Exact=0.300,  ±1=0.911,  Bin=1.000
MST 5:  N=167,  Exact=0.653,  ±1=1.000,  Bin=1.000
MST 6:  N=154,  Exact=0.117,  ±1=0.682,  Bin=1.000
MST 7:  N=65,  Exact=0.031,  ±1=0.123,  Bin=1.000
MST 8:  N=130,  Exact=0.000,  ±1=0.038,  Bin=0.000
MST 9:  N=124,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 10:  N=110,  Exact=0.000,  ±1=0.000,  Bin=0.000


### Casual Conversation v2

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set)
model_path = r"G:\Thesis\Models\RandomForest\Model 2\rf_reg_bins256_RMSE1.323_ACC050.346_20251129_215235.joblib"
csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\rf_mst_predictions.csv"
split_json = r"G:\Thesis\Models\RandomForest\Model 2\train_val_split.json"

metrics, per_tone, df_rf_val = evaluate_rf_model(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    file_path_column="filename",
    label_column="mst_label",
    person_id_column="subject_id",
    split_json=split_json,
    split_key="val_persons",
    output_csv=output_csv
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
            f"Exact={stats['exact']:.3f},  "
            f"±1={stats['pm1']:.3f},  "
            f"Bin={stats['bin']:.3f}")

[INFO] Loading RF model from G:\Thesis\Models\RandomForest\Model 2\rf_reg_bins256_RMSE1.323_ACC050.346_20251129_215235.joblib
[INFO] Loading split from: G:\Thesis\Models\RandomForest\Model 2\train_val_split.json
[INFO] Filtered from 184201 → 64876 images using split 'val_persons'
[INFO] Detected model type: Regressor


Evaluating RF: 100%|██████████| 64876/64876 [1:49:28<00:00,  9.88it/s]  


[INFO] Saved predictions to G:\Thesis\Models\RandomForest\Model 2\rf_mst_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.3463
±1 tolerance accuracy: 0.7562
3-bin accuracy:        0.7001

=========== PER-TONE RESULTS ===========
MST 1:  N=460,  Exact=0.000,  ±1=0.000,  Bin=0.222
MST 2:  N=4923,  Exact=0.000,  ±1=0.050,  Bin=0.050
MST 3:  N=12052,  Exact=0.053,  ±1=0.737,  Bin=0.053
MST 4:  N=13924,  Exact=0.621,  ±1=0.978,  Bin=0.962
MST 5:  N=20427,  Exact=0.610,  ±1=0.989,  Bin=0.990
MST 6:  N=8614,  Exact=0.083,  ±1=0.681,  Bin=0.992
MST 7:  N=2278,  Exact=0.002,  ±1=0.091,  Bin=0.997
MST 8:  N=1536,  Exact=0.000,  ±1=0.015,  Bin=0.000
MST 9:  N=614,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 10:  N=48,  Exact=0.000,  ±1=0.000,  Bin=0.000


### FACET

In [ ]:
# Testing on FACET Dataset
model_path = r"G:\Thesis\Models\RandomForest\Model 2\rf_reg_bins256_RMSE1.323_ACC050.346_20251129_215235.joblib"
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\rf_mst_predictions.csv"

metrics, per_tone, df = evaluate_rf_model(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    file_path_column="filename",
    label_column="mst_label",
    bins=256,
    output_csv=output_csv
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}:  N={stats['count']},  "
            f"Exact={stats['exact']:.3f},  "
            f"±1={stats['pm1']:.3f},  "
            f"Bin={stats['bin']:.3f}")


[INFO] Loading RF model from G:\Thesis\Models\RandomForest\Model 2\rf_reg_bins256_RMSE1.323_ACC050.346_20251129_215235.joblib
[INFO] No split JSON provided, using all 2677 images
[INFO] Detected model type: Regressor


Evaluating RF: 100%|██████████| 2677/2677 [04:39<00:00,  9.59it/s]


[INFO] Saved predictions to G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\rf_mst_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.1759
±1 tolerance accuracy: 0.5499
3-bin accuracy:        0.4841

=========== PER-TONE RESULTS ===========
MST 1:  N=70,  Exact=0.000,  ±1=0.000,  Bin=0.071
MST 2:  N=559,  Exact=0.000,  ±1=0.052,  Bin=0.052
MST 3:  N=693,  Exact=0.036,  ±1=0.593,  Bin=0.036
MST 4:  N=485,  Exact=0.491,  ±1=0.984,  Bin=0.996
MST 5:  N=349,  Exact=0.582,  ±1=0.991,  Bin=0.991
MST 6:  N=288,  Exact=0.017,  ±1=0.715,  Bin=1.000
MST 7:  N=120,  Exact=0.000,  ±1=0.025,  Bin=1.000
MST 8:  N=67,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 9:  N=41,  Exact=0.000,  ±1=0.000,  Bin=0.000
MST 10:  N=5,  Exact=0.000,  ±1=0.000,  Bin=0.000


## DenseNet 121 Approach

In [19]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from PIL import Image
import json  # NEW

import torch
import torch.nn as nn
from torchvision import transforms
from skimage.color import rgb2lab

# =====================================================================
# IMPORT MODEL + UTILITIES FROM YOUR TRAINING FILE
# =====================================================================
from DenseNet121_SkinTone_Training import (
    DenseNet121LabImproved,
    monk_scalar_to_lab
)

# =====================================================================
# BASIC METRICS
# =====================================================================
def to_bin(x):
    if 1 <= x <= 3:
        return 0
    elif 4 <= x <= 7:
        return 1
    else:
        return 2

def accuracy_exact(y_true, y_pred):
    return (y_true == y_pred).mean()

def accuracy_pm1(y_true, y_pred):
    return (np.abs(y_true - y_pred) <= 1).mean()

def accuracy_bins(y_true, y_pred):
    return (np.array([to_bin(x) for x in y_true]) ==
            np.array([to_bin(x) for x in y_pred])).mean()

# =====================================================================
# LAB Transform (same normalisation as training)
# =====================================================================
class EvalLABTransform:
    def __init__(self, lab_mean, lab_std):
        self.resize = transforms.Resize((224, 224))
        self.lab_mean = lab_mean.astype(np.float32)
        self.lab_std  = lab_std.astype(np.float32)

    def __call__(self, img_pil):
        img_pil = self.resize(img_pil)
        rgb = np.asarray(img_pil).astype(np.float32) / 255.0
        lab = rgb2lab(rgb).astype(np.float32)
        lab_norm = (lab - self.lab_mean) / self.lab_std
        return torch.from_numpy(lab_norm.transpose(2, 0, 1)).float()

# =====================================================================
# REGRESSION: compute LAB L2 error
# =====================================================================
def l2_lab(pred_scalar, true_scalar):
    """
    pred_scalar, true_scalar in [1,10] MST scale.
    Convert to LAB and compute L2 distance.
    """
    p_lab = monk_scalar_to_lab([pred_scalar])[0]
    t_lab = monk_scalar_to_lab([true_scalar])[0]
    return float(np.sqrt(np.sum((p_lab - t_lab) ** 2)))

# =====================================================================
# MAIN EVALUATION FUNCTION
# =====================================================================
def evaluate_densenet_model(
    model_path,
    csv_path,
    image_root,
    lab_mean,
    lab_std,
    mode="classification",
    file_path_column="filename",
    label_column="label",
    output_csv="densenet_predictions.csv",
    device="cuda",
    # NEW: split-json-based filtering (train/val/etc.)
    person_id_column="person_id",
    split_json=None,     # path to split JSON file
    split_key=None       # key inside JSON, e.g. "val_persons" or "train_persons"
):

    assert mode in {"classification", "regression"}

    print(f"[INFO] Evaluating DenseNet121 model from: {model_path}")
    print(f"[INFO] Mode: {mode}")
    device = torch.device(device)

    # --------------------------------------------------------------
    # Load model
    # --------------------------------------------------------------
    model = DenseNet121LabImproved(mode=mode, finetune_mode="all")
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    # --------------------------------------------------------------
    # Dataset
    # --------------------------------------------------------------
    df = pd.read_csv(csv_path)

    # --------------------------------------------------------------
    # OPTIONAL: Filter dataset based on split JSON (train/val)
    # --------------------------------------------------------------
    if split_json is not None:
        print(f"[INFO] Loading split from: {split_json}")
        with open(split_json, "r") as f:
            split_data = json.load(f)

        if split_key is None:
            raise ValueError("split_json was provided but split_key is None. "
                             "Pass e.g. split_key='val_persons' or 'train_persons'.")

        if split_key not in split_data:
            raise ValueError(f"split_key '{split_key}' not found in split JSON. "
                             f"Available keys: {list(split_data.keys())}")

        # Depending on how you stored IDs, you can treat them as int or str.
        # To mirror your VGG16 logic, cast to int:
        try:
            person_ids = [int(x) for x in split_data[split_key]]
            df_ids = df[person_id_column].astype(int)
        except Exception:
            # Fallback: treat as string IDs
            person_ids = [str(x) for x in split_data[split_key]]
            df_ids = df[person_id_column].astype(str)

        original_count = len(df)
        df = df[df_ids.isin(person_ids)].reset_index(drop=True)
        print(f"[INFO] Filtered from {original_count} to {len(df)} images using split '{split_key}'")
    else:
        print(f"[INFO] No split JSON provided, using all {len(df)} images")

    results = []

    transform = EvalLABTransform(lab_mean, lab_std)

    # --------------------------------------------------------------
    # Evaluation loop
    # --------------------------------------------------------------
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating DenseNet"):
        img_name = row[file_path_column]
        true_label = float(row[label_column])

        img_path = os.path.join(image_root, img_name)
        if not os.path.exists(img_path):
            print(f"[WARN] Missing: {img_path}")
            continue

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            print(f"[WARN] Failed: {img_path}")
            continue

        img_t = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            out = model(img_t)

            if mode == "classification":
                pred_idx = torch.argmax(out, dim=1).item()        # 0..9
                pred_mst = pred_idx + 1                           # 1..10
                # scalar in [0,1] for consistency with regression metrics
                pred_scalar = (pred_mst - 1) / 9.0

            else:  # regression (model outputs [0,1])
                pred_scalar = float(out.item())
                pred_mst = int(np.clip(round(pred_scalar * 9 + 1), 1, 10))

        abs_err = abs(true_label - pred_mst)
        lab_err = l2_lab(pred_mst, true_label)

        results.append({
            "image_path": img_path,
            "true_label": int(true_label),
            "pred_scalar": pred_scalar,
            "pred_label": pred_mst,
            "abs_error": abs_err,
            "l2_lab": lab_err,
            "match_exact": 1 if true_label == pred_mst else 0,
            "match_pm1": 1 if abs_err <= 1 else 0,
            "true_bin": to_bin(int(true_label)),
            "pred_bin": to_bin(int(pred_mst)),
            "match_bin": 1 if to_bin(int(true_label)) == to_bin(int(pred_mst)) else 0
        })

    # Convert to DataFrame
    df_out = pd.DataFrame(results)
    df_out.to_csv(output_csv, index=False)
    print(f"[INFO] Saved predictions → {output_csv}")

    # ------------------------------------------------------------------
    # GLOBAL METRICS
    # ------------------------------------------------------------------
    y_true = df_out["true_label"].values
    y_pred = df_out["pred_label"].values

    metrics = {
        "accuracy_exact": accuracy_exact(y_true, y_pred),
        "accuracy_pm1": accuracy_pm1(y_true, y_pred),
        "accuracy_3bins": accuracy_bins(y_true, y_pred),
        "mean_l2_lab": df_out["l2_lab"].mean()
    }

    # ------------------------------------------------------------------
    # PER-TONE PERFORMANCE
    # ------------------------------------------------------------------
    per_tone = {}
    for tone in range(1, 11):
        subset = df_out[df_out["true_label"] == tone]
        if len(subset) == 0:
            continue

        per_tone[tone] = {
            "count": len(subset),
            "exact": subset["match_exact"].mean(),
            "pm1": subset["match_pm1"].mean(),
            "bin": subset["match_bin"].mean(),
            "l2_lab_mean": subset["l2_lab"].mean()
        }

    return metrics, per_tone, df_out


### Monk Skin Tone Dataset

In [37]:
# Testing on Monk Skin Tone Dataset
# PRECOMPUTED LAB STATS
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

model_path = r"G:\Thesis\Models\DenseNet_LAB\Model 1\densenet121_lab_best.pth"
csv_path = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\densenet_predictions.csv"

metrics, per_tone, df_preds = evaluate_densenet_model(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    lab_mean=lab_mean,
    lab_std=lab_std,
    mode="classification",                    
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")
print(f"Mean L2-LAB distance:  {metrics['mean_l2_lab']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}, "
            f"L2={stats['l2_lab_mean']:.3f}")

[INFO] Evaluating DenseNet121 model from: G:\Thesis\Models\DenseNet_LAB\Model 1\densenet121_lab_best.pth
[INFO] Mode: classification
[INFO] No split JSON provided, using all 1388 images


Evaluating DenseNet: 100%|██████████| 1388/1388 [00:50<00:00, 27.60it/s]

[INFO] Saved predictions → G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\densenet_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.2104
±1 tolerance accuracy: 0.5173
3-bin accuracy:        0.5641
Mean L2-LAB distance:  20.0858

=========== PER-TONE RESULTS ===========
MST 1: N=174, Exact=0.011, ±1=0.471, Bin=0.718, L2=10.706
MST 2: N=198, Exact=0.167, ±1=0.237, Bin=0.237, L2=20.994
MST 3: N=86, Exact=0.023, ±1=0.407, Bin=0.035, L2=31.176
MST 4: N=180, Exact=0.394, ±1=0.567, Bin=0.822, L2=20.575
MST 5: N=167, Exact=0.102, ±1=0.683, Bin=0.749, L2=24.068
MST 6: N=154, Exact=0.273, ±1=0.500, Bin=0.513, L2=18.251
MST 7: N=65, Exact=0.338, ±1=0.938, Bin=0.462, L2=10.612
MST 8: N=130, Exact=0.746, ±1=0.885, Bin=0.762, L2=6.497
MST 9: N=124, Exact=0.040, ±1=0.645, Bin=0.645, L2=23.232
MST 10: N=110, Exact=0.009, ±1=0.045, Bin=0.427, L2=38.450


### Casual Conversation v2

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set)
# PRECOMPUTED LAB STATS
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

model_path = r"G:\Thesis\Models\DenseNet_LAB\Model 1\densenet121_lab_best.pth"
csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\densenet_predictions.csv"
split_json = r"G:\Thesis\Models\DenseNet_LAB\Model 1\train_val_split.json"

metrics, per_tone, df_preds = evaluate_densenet_model(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    lab_mean=lab_mean,
    lab_std=lab_std,
    mode="classification",                    
    file_path_column="filename",
    label_column="mst_label",
    person_id_column="subject_id",
    split_json=split_json,
    split_key="val_persons",
    output_csv=output_csv,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")
print(f"Mean L2-LAB distance:  {metrics['mean_l2_lab']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}, "
            f"L2={stats['l2_lab_mean']:.3f}")

### FACET

In [20]:
# Testing on FACET Dataset
# PRECOMPUTED LAB STATS
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

model_path = r"G:\Thesis\Models\DenseNet_LAB\Model 1\densenet121_lab_best.pth"
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\densenet_predictions.csv"

metrics, per_tone, df_preds = evaluate_densenet_model(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    lab_mean=lab_mean,
    lab_std=lab_std,
    mode="classification",                    
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")
print(f"Mean L2-LAB distance:  {metrics['mean_l2_lab']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}, "
            f"L2={stats['l2_lab_mean']:.3f}")

[INFO] Evaluating DenseNet121 model from: G:\Thesis\Models\DenseNet_LAB\Model 1\densenet121_lab_best.pth
[INFO] Mode: classification
[INFO] No split JSON provided, using all 2677 images


Evaluating DenseNet: 100%|██████████| 2677/2677 [01:41<00:00, 26.33it/s]


[INFO] Saved predictions → G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\densenet_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.2099
±1 tolerance accuracy: 0.5290
3-bin accuracy:        0.5760
Mean L2-LAB distance:  20.3659

=========== PER-TONE RESULTS ===========
MST 1: N=70, Exact=0.057, ±1=0.514, Bin=0.600, L2=16.598
MST 2: N=559, Exact=0.345, ±1=0.497, Bin=0.497, L2=18.373
MST 3: N=693, Exact=0.108, ±1=0.505, Bin=0.413, L2=21.530
MST 4: N=485, Exact=0.157, ±1=0.412, Bin=0.643, L2=22.817
MST 5: N=349, Exact=0.172, ±1=0.524, Bin=0.762, L2=23.247
MST 6: N=288, Exact=0.267, ±1=0.694, Bin=0.767, L2=17.803
MST 7: N=120, Exact=0.350, ±1=0.808, Bin=0.650, L2=13.966
MST 8: N=67, Exact=0.493, ±1=0.746, Bin=0.537, L2=15.155
MST 9: N=41, Exact=0.049, ±1=0.512, Bin=0.512, L2=25.300
MST 10: N=5, Exact=0.000, ±1=0.200, Bin=0.400, L2=26.319


## VGG16 Approach

In [21]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from PIL import Image
import json

import torch
import torch.nn as nn
from torchvision import transforms
from skimage.color import rgb2lab

###############################################################
# 1. Import the model class from your training script
###############################################################

from VGG16_Reg_Cla import (
    VGG16MSTClassifier,
    VGG16MSTRegressor
)


###############################################################
# 2. RGB Transform (same as training)
###############################################################

def rgb_eval_transform():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])


###############################################################
# 3. LAB Transform (same as training)
###############################################################

class EvalLABTransform:
    def __init__(self, lab_mean, lab_std):
        self.resize = transforms.Resize((224, 224))
        self.lab_mean = lab_mean.astype(np.float32)
        self.lab_std = lab_std.astype(np.float32)

    def __call__(self, img_pil):

        img = self.resize(img_pil)
        rgb = np.asarray(img).astype(np.float32) / 255.0

        lab = rgb2lab(rgb).astype(np.float32)
        lab_norm = (lab - self.lab_mean) / self.lab_std

        # Convert HWC → CHW
        return torch.from_numpy(lab_norm.transpose(2, 0, 1)).float()


###############################################################
# 4. HYBRID Transform (RGB + LAB concatenated)
###############################################################

class EvalHybridTransform:
    def __init__(self, lab_mean, lab_std):
        self.resize = transforms.Resize((224, 224))
        self.lab_mean = lab_mean.astype(np.float32)
        self.lab_std = lab_std.astype(np.float32)

        self.rgb_norm = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __call__(self, img_pil):

        img = self.resize(img_pil)
        rgb_arr = np.asarray(img).astype(np.float32) / 255.0

        ###############
        # RGB BRANCH
        ###############
        rgb_tensor = torch.from_numpy(rgb_arr.transpose(2, 0, 1)).float()
        rgb_tensor = self.rgb_norm(rgb_tensor)

        ###############
        # LAB BRANCH
        ###############
        lab_arr = rgb2lab(rgb_arr).astype(np.float32)
        lab_norm = (lab_arr - self.lab_mean) / self.lab_std
        lab_tensor = torch.from_numpy(lab_norm.transpose(2, 0, 1)).float()

        ###############
        # CONCAT: [RGB||LAB] → 6 channels
        ###############
        return torch.cat([rgb_tensor, lab_tensor], dim=0)


###############################################################
# 5. Metric Functions (updated for mst3 mode)
###############################################################

def mst_to_bin(mst):
    """3-bin grouping: Light/Mid/Dark (1-3 / 4-7 / 8-10)"""
    if mst <= 3: 
        return 0  # Light
    elif mst <= 7: 
        return 1  # Mid
    else:
        return 2  # Dark


def class_to_mst(pred_class, label_mode):
    """
    Convert predicted class index to MST label(s).
    
    For mst10: class 0-9 → MST 1-10
    For mst3:  class 0-2 → bin (we return the bin center for display)
    """
    if label_mode == "mst10":
        return pred_class + 1  # Direct mapping
    
    elif label_mode == "mst3":
        # Class 0 → light (1-3), class 1 → mid (4-7), class 2 → dark (8-10)
        # For display purposes, return bin ID (0, 1, 2)
        return pred_class
    
    else:
        raise ValueError(f"Unknown label_mode: {label_mode}")


def mst_to_class(mst, label_mode):
    """Convert MST label to class index for comparison."""
    if label_mode == "mst10":
        return mst - 1
    elif label_mode == "mst3":
        return mst_to_bin(mst)
    else:
        raise ValueError(f"Unknown label_mode: {label_mode}")


def accuracy_exact(y_true, y_pred, label_mode):
    """
    For mst10: exact MST match
    For mst3: exact bin match
    """
    if label_mode == "mst10":
        return (y_true == y_pred).mean()
    elif label_mode == "mst3":
        true_bins = np.array([mst_to_bin(t) for t in y_true])
        pred_bins = y_pred  # Already in bin format
        return (true_bins == pred_bins).mean()


def accuracy_pm1(y_true, y_pred, label_mode):
    """
    For mst10: ±1 MST tolerance
    For mst3: ±1 bin tolerance (though less meaningful with only 3 bins)
    """
    if label_mode == "mst10":
        return (np.abs(y_true - y_pred) <= 1).mean()
    elif label_mode == "mst3":
        true_bins = np.array([mst_to_bin(t) for t in y_true])
        pred_bins = y_pred
        return (np.abs(true_bins - pred_bins) <= 1).mean()


def accuracy_bins(y_true, y_pred, label_mode):
    """
    3-bin accuracy (same as exact for mst3 mode)
    """
    true_bins = np.array([mst_to_bin(t) for t in y_true])
    
    if label_mode == "mst10":
        pred_bins = np.array([mst_to_bin(p) for p in y_pred])
    else:  # mst3
        pred_bins = y_pred
    
    return (true_bins == pred_bins).mean()


###############################################################
# 6. Main Evaluation Function (updated for mst3 and split JSON)
###############################################################

def evaluate_vgg16_classifier(
    model_path,
    csv_path,
    image_root,
    label_mode="mst10",               # "mst10" or "mst3"
    input_mode="rgb",                 # "rgb", "lab", or "hybrid"
    lab_mean=None,
    lab_std=None,
    file_path_column="filename",
    label_column="label",
    person_id_column="person_id",     # person_id column name
    split_json=None,                  # path to train_val_split.json
    split_key=None,                   # "train" or "val"
    output_csv="mst_predictions.csv",
    device="cuda"
):

    print(f"[INFO] Evaluating model: {model_path}")
    print(f"[INFO] Label mode: {label_mode}")
    print(f"[INFO] Input mode: {input_mode}")
    device = torch.device(device)

    ###############################################################
    # Load model (with correct input mode and num_classes)
    ###############################################################
    num_classes = 10 if label_mode == "mst10" else 3
    
    model = VGG16MSTClassifier(
        input_mode=input_mode,
        num_classes=num_classes
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    ###############################################################
    # Load dataset and filter based on split
    ###############################################################
    df = pd.read_csv(csv_path)
    
    # Filter by train/val split if JSON provided
    if split_json is not None:
        print(f"[INFO] Loading split from: {split_json}")
        
        with open(split_json, 'r') as f:
            split_data = json.load(f)
        
        if split_key != None:
            person_ids = [int(x) for x in split_data[split_key]]
        else:
            raise ValueError(f"split_key not a valid key, got: {split_key}")
        
        # Filter dataframe to only include specified person_ids
        original_count = len(df)
        df = df[df[person_id_column].isin(person_ids)].reset_index(drop=True)
        print(f"[INFO] Filtered from {original_count} to {len(df)} images")
    else:
        print(f"[INFO] No split JSON provided, using all {len(df)} images")
    
    results = []

    ###############################################################
    # Select transform
    ###############################################################
    if input_mode == "rgb":
        transform = rgb_eval_transform()

    elif input_mode == "lab":
        if lab_mean is None or lab_std is None:
            raise ValueError("LAB mode requires lab_mean and lab_std")
        transform = EvalLABTransform(lab_mean, lab_std)

    elif input_mode == "hybrid":
        if lab_mean is None or lab_std is None:
            raise ValueError("Hybrid mode requires lab_mean and lab_std")
        transform = EvalHybridTransform(lab_mean, lab_std)

    else:
        raise ValueError(f"Unknown input_mode: {input_mode}")

    ###############################################################
    # Iterate over images and run inference
    ###############################################################
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating"):
        img_path = os.path.join(image_root, row[file_path_column])
        true_mst = int(row[label_column])  # MST 1–10

        if not os.path.exists(img_path):
            print(f"[WARN] Missing file: {img_path}")
            continue

        try:
            img_pil = Image.open(img_path).convert("RGB")
        except:
            print(f"[WARN] Failed to open: {img_path}")
            continue

        # Transform
        img_tensor = transform(img_pil).unsqueeze(0).to(device)

        # Predict logits → classes
        with torch.no_grad():
            logits = model(img_tensor)
            pred_class = logits.argmax(dim=1).item()

        # Convert to appropriate output format
        pred_output = class_to_mst(pred_class, label_mode)
        true_class = mst_to_class(true_mst, label_mode)
        
        # Compute bin assignments
        true_bin = mst_to_bin(true_mst)
        
        if label_mode == "mst10":
            pred_mst = pred_output
            pred_bin = mst_to_bin(pred_mst)
            abs_err = abs(true_mst - pred_mst)
        else:  # mst3
            pred_bin = pred_output  # Already a bin
            pred_mst = None  # Not directly predicted
            abs_err = abs(true_bin - pred_bin)

        results.append({
            "image_path": img_path,
            "true_mst": true_mst,
            "true_class": true_class,
            "true_bin": true_bin,
            "pred_class": pred_class,
            "pred_mst": pred_mst if label_mode == "mst10" else None,
            "pred_bin": pred_bin,
            "abs_error": abs_err,
            "match_exact": 1 if pred_class == true_class else 0,
            "match_pm1": 1 if abs_err <= 1 else 0,
            "match_bin": 1 if pred_bin == true_bin else 0
        })

    ###############################################################
    # Save CSV
    ###############################################################
    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv, index=False)
    print(f"[INFO] Saved predictions: {output_csv}")

    ###############################################################
    # Compute global metrics
    ###############################################################
    y_true_mst = out_df["true_mst"].values
    
    if label_mode == "mst10":
        y_pred = out_df["pred_mst"].values
    else:  # mst3
        y_pred = out_df["pred_bin"].values

    metrics = {
        "accuracy_exact": accuracy_exact(y_true_mst, y_pred, label_mode),
        "accuracy_pm1": accuracy_pm1(y_true_mst, y_pred, label_mode),
        "accuracy_3bins": accuracy_bins(y_true_mst, y_pred, label_mode)
    }

    ###############################################################
    # Per tone results (for mst10) or per bin (for mst3)
    ###############################################################
    per_tone = {}
    
    if label_mode == "mst10":
        # Per-tone metrics (MST 1-10)
        for tone in range(1, 11):
            subset = out_df[out_df["true_mst"] == tone]
            if len(subset) == 0:
                continue

            per_tone[tone] = {
                "count": len(subset),
                "exact": subset["match_exact"].mean(),
                "pm1": subset["match_pm1"].mean(),
                "bin": subset["match_bin"].mean(),
            }
    else:  # mst3
        # Per-bin metrics (bins 0-2)
        bin_names = {0: "Light (1-3)", 1: "Mid (4-7)", 2: "Dark (8-10)"}
        for bin_id in range(3):
            subset = out_df[out_df["true_bin"] == bin_id]
            if len(subset) == 0:
                continue

            per_tone[bin_names[bin_id]] = {
                "count": len(subset),
                "exact": subset["match_exact"].mean(),
                "pm1": subset["match_pm1"].mean(),
                "bin": subset["match_bin"].mean(),
            }

    return metrics, per_tone, out_df


###############################################################
# 7. Regression Model Evaluation Function
###############################################################

def evaluate_vgg16_regressor(
    model_path,
    csv_path,
    image_root,
    label_mode="mst10",               # "mst10" or "mst3"
    input_mode="rgb",                 # "rgb", "lab", or "hybrid"
    output_range="sigmoid",            # "direct" (outputs 1-10) or "sigmoid" (outputs 0-1, needs scaling)
    lab_mean=None,
    lab_std=None,
    file_path_column="filename",
    label_column="label",
    person_id_column="person_id",
    split_json=None,
    split_key=None,
    output_csv="mst_predictions_regressor.csv",
    device="cuda"
):
    """
    Evaluate VGG16 Regressor model.
    
    The model outputs continuous values which are processed based on output_range:
    - "direct": Model outputs MST values directly (1-10 range), rounded and clipped
    - "sigmoid": Model outputs sigmoid [0,1], scaled to MST via: pred * 9 + 1
    """

    print(f"[INFO] Evaluating REGRESSOR model: {model_path}")
    print(f"[INFO] Label mode: {label_mode}")
    print(f"[INFO] Input mode: {input_mode}")
    print(f"[INFO] Output range: {output_range}")
    device = torch.device(device)

    ###############################################################
    # Load model
    ###############################################################
    model = VGG16MSTRegressor(input_mode=input_mode)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    ###############################################################
    # Load dataset and filter based on split
    ###############################################################
    df = pd.read_csv(csv_path)
    
    if split_json is not None:
        print(f"[INFO] Loading split from: {split_json}")
        
        with open(split_json, 'r') as f:
            split_data = json.load(f)
        
        if split_key != None:
            person_ids = [int(x) for x in split_data[split_key]]
        else:
            raise ValueError(f"split_key not a valid key, got: {split_key}")
        
        original_count = len(df)
        df = df[df[person_id_column].isin(person_ids)].reset_index(drop=True)
        print(f"[INFO] Filtered from {original_count} to {len(df)} images")
    else:
        print(f"[INFO] No split JSON provided, using all {len(df)} images")
    
    results = []

    ###############################################################
    # Select transform
    ###############################################################
    if input_mode == "rgb":
        transform = rgb_eval_transform()
    elif input_mode == "lab":
        if lab_mean is None or lab_std is None:
            raise ValueError("LAB mode requires lab_mean and lab_std")
        transform = EvalLABTransform(lab_mean, lab_std)
    elif input_mode == "hybrid":
        if lab_mean is None or lab_std is None:
            raise ValueError("Hybrid mode requires lab_mean and lab_std")
        transform = EvalHybridTransform(lab_mean, lab_std)
    else:
        raise ValueError(f"Unknown input_mode: {input_mode}")

    ###############################################################
    # Iterate over images and run inference
    ###############################################################
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating Regressor"):
        img_path = os.path.join(image_root, row[file_path_column])
        true_mst = int(row[label_column])  # MST 1–10

        if not os.path.exists(img_path):
            print(f"[WARN] Missing file: {img_path}")
            continue

        try:
            img_pil = Image.open(img_path).convert("RGB")
        except:
            print(f"[WARN] Failed to open: {img_path}")
            continue

        # Transform
        img_tensor = transform(img_pil).unsqueeze(0).to(device)

        # Predict continuous value
        with torch.no_grad():
            pred_raw = model(img_tensor).squeeze().item()
        
        # Process prediction based on output range
        if output_range == "sigmoid":
            # Sigmoid output [0,1] → scale to MST [1,10]
            pred_scaled = pred_raw * 9 + 1
            pred_mst_rounded = round(pred_scaled)
            pred_mst_clipped = int(np.clip(pred_mst_rounded, 1, 10))
        elif output_range == "direct":
            # Direct MST output → just round and clip
            pred_mst_rounded = round(pred_raw)
            pred_mst_clipped = int(np.clip(pred_mst_rounded, 1, 10))
        else:
            raise ValueError(f"Unknown output_range: {output_range}. Use 'direct' or 'sigmoid'.")
        
        # Compute bin assignments
        true_bin = mst_to_bin(true_mst)
        pred_bin = mst_to_bin(pred_mst_clipped)
        
        # Compute absolute error on MST scale
        abs_err = abs(true_mst - pred_mst_clipped)
        
        # For label_mode handling
        if label_mode == "mst10":
            pred_output = pred_mst_clipped
            true_class = true_mst - 1  # For consistency
            pred_class = pred_mst_clipped - 1
        else:  # mst3
            pred_output = pred_bin
            true_class = true_bin
            pred_class = pred_bin
            abs_err = abs(true_bin - pred_bin)  # Recompute for bin scale

        results.append({
            "image_path": img_path,
            "true_label": true_mst,  # Keep naming consistent with old code
            "true_mst": true_mst,
            "true_class": true_class,
            "true_bin": true_bin,
            "pred_scalar": pred_raw,  # Raw model output
            "pred_mst_rounded": pred_mst_rounded,  # Before clipping
            "pred_label": pred_mst_clipped,  # Final prediction (like old code)
            "pred_mst": pred_mst_clipped,
            "pred_class": pred_class,
            "pred_bin": pred_bin,
            "abs_error": abs_err,
            "match_exact": 1 if pred_mst_clipped == true_mst else 0,
            "match_pm1": 1 if abs(true_mst - pred_mst_clipped) <= 1 else 0,
            "match_bin": 1 if pred_bin == true_bin else 0
        })

    ###############################################################
    # Save CSV
    ###############################################################
    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv, index=False)
    print(f"[INFO] Saved predictions → {output_csv}")

    ###############################################################
    # Compute global metrics
    ###############################################################
    y_true = out_df["true_label"].values
    y_pred = out_df["pred_label"].values

    metrics = {
        "accuracy_exact": accuracy_exact(y_true, y_pred, label_mode),
        "accuracy_pm1": accuracy_pm1(y_true, y_pred, label_mode),
        "accuracy_3bins": accuracy_bins(y_true, y_pred, label_mode),
        "mae": np.mean(np.abs(y_true - y_pred)),
        "rmse": np.sqrt(np.mean((y_true - y_pred) ** 2))
    }

    ###############################################################
    # Per tone results
    ###############################################################
    per_tone = {}
    
    if label_mode == "mst10":
        # Per-tone metrics (MST 1-10)
        for tone in range(1, 11):
            subset = out_df[out_df["true_label"] == tone]
            if len(subset) == 0:
                continue

            per_tone[tone] = {
                "count": len(subset),
                "exact": subset["match_exact"].mean(),
                "pm1": subset["match_pm1"].mean(),
                "bin": subset["match_bin"].mean(),
                "mae": np.mean(np.abs(subset["true_label"].values - subset["pred_label"].values)),
            }
    else:  # mst3
        # Per-bin metrics (bins 0-2)
        bin_names = {0: "Light (1-3)", 1: "Mid (4-7)", 2: "Dark (8-10)"}
        for bin_id in range(3):
            subset = out_df[out_df["true_bin"] == bin_id]
            if len(subset) == 0:
                continue

            per_tone[bin_names[bin_id]] = {
                "count": len(subset),
                "exact": subset["match_exact"].mean(),
                "pm1": subset["match_pm1"].mean(),
                "bin": subset["match_bin"].mean(),
                "mae": np.mean(np.abs(subset["true_label"].values - subset["pred_label"].values)),
            }

    return metrics, per_tone, out_df

### Model 1

### Monk Skin Tone Dataset

In [42]:
# Testing on Monk Skin Tone Dataset
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 1\vgg16_lab_best.pth"
csv_path = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv=r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\vgg16_model_1_predictions.csv"

metrics, per_tone, df = evaluate_vgg16_regressor(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    output_range="sigmoid",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv,
    device="cuda",
    split_json=None
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")


[INFO] Evaluating REGRESSOR model: G:\Thesis\CasualConversationv2_Dataset\Models\Model 1\vgg16_lab_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] Output range: sigmoid
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 1388 images


Evaluating Regressor: 100%|██████████| 1388/1388 [00:34<00:00, 39.88it/s]

[INFO] Saved predictions → G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\vgg16_model_1_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.2277
±1 tolerance accuracy: 0.6203
3-bin accuracy:        0.6729

=========== PER-TONE RESULTS ===========
MST 1: N=174, Exact=0.029, ±1=0.310, Bin=0.695
MST 2: N=198, Exact=0.086, ±1=0.510, Bin=0.510
MST 3: N=86, Exact=0.035, ±1=0.302, Bin=0.035
MST 4: N=180, Exact=0.400, ±1=0.794, Bin=0.889
MST 5: N=167, Exact=0.323, ±1=0.832, Bin=0.940
MST 6: N=154, Exact=0.364, ±1=0.909, Bin=0.955
MST 7: N=65, Exact=0.508, ±1=0.985, Bin=0.723
MST 8: N=130, Exact=0.523, ±1=0.892, Bin=0.615
MST 9: N=124, Exact=0.065, ±1=0.605, Bin=0.605
MST 10: N=110, Exact=0.000, ±1=0.027, Bin=0.391


### Casual Conversation v2

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set)
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 1\vgg16_lab_best.pth"
split_json = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 1\train_val_split.json"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\vgg16_model_1_predictions.csv"

metrics3, per_bin, df3 = evaluate_vgg16_regressor(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    output_range="sigmoid",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    person_id_column="subject_id",
    split_json=split_json,
    split_key="val_persons",
    output_csv=output_csv,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

### FACET

In [22]:
# Testing on FACET Dataset
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 1\vgg16_lab_best.pth"
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv=r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\vgg16_model_1_predictions.csv"

metrics, per_tone, df = evaluate_vgg16_regressor(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    output_range="sigmoid",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv,
    device="cuda",
    split_json=None
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating REGRESSOR model: G:\Thesis\CasualConversationv2_Dataset\Models\Model 1\vgg16_lab_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] Output range: sigmoid
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 2677 images


Evaluating Regressor: 100%|██████████| 2677/2677 [00:43<00:00, 61.98it/s]


[INFO] Saved predictions → G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\vgg16_model_1_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.2682
±1 tolerance accuracy: 0.6694
3-bin accuracy:        0.5999

=========== PER-TONE RESULTS ===========
MST 1: N=70, Exact=0.029, ±1=0.143, Bin=0.471
MST 2: N=559, Exact=0.138, ±1=0.456, Bin=0.456
MST 3: N=693, Exact=0.277, ±1=0.716, Bin=0.361
MST 4: N=485, Exact=0.392, ±1=0.819, Bin=0.751
MST 5: N=349, Exact=0.324, ±1=0.788, Bin=0.840
MST 6: N=288, Exact=0.299, ±1=0.740, Bin=0.934
MST 7: N=120, Exact=0.317, ±1=0.733, Bin=0.842
MST 8: N=67, Exact=0.254, ±1=0.582, Bin=0.299
MST 9: N=41, Exact=0.073, ±1=0.439, Bin=0.439
MST 10: N=5, Exact=0.000, ±1=0.200, Bin=0.600


### Model 2

### Monk Skin Tone Dataset

In [43]:
# Testing on Monk Skin Tone Dataset
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 2\vgg16_lab_best.pth"
csv_path = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\vgg16_model_2_predictions.csv"

metrics, per_tone, df = evaluate_vgg16_regressor(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    output_range="sigmoid",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv,
    device="cuda",
    split_json=None
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating REGRESSOR model: G:\Thesis\CasualConversationv2_Dataset\Models\Model 2\vgg16_lab_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] Output range: sigmoid
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 1388 images


Evaluating Regressor: 100%|██████████| 1388/1388 [00:35<00:00, 39.58it/s]

[INFO] Saved predictions → G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\vgg16_model_2_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.1859
±1 tolerance accuracy: 0.4928
3-bin accuracy:        0.5389

=========== PER-TONE RESULTS ===========
MST 1: N=174, Exact=0.092, ±1=0.310, Bin=0.523
MST 2: N=198, Exact=0.197, ±1=0.470, Bin=0.470
MST 3: N=86, Exact=0.151, ±1=0.477, Bin=0.360
MST 4: N=180, Exact=0.178, ±1=0.761, Bin=0.667
MST 5: N=167, Exact=0.449, ±1=0.826, Bin=0.898
MST 6: N=154, Exact=0.221, ±1=0.649, Bin=0.792
MST 7: N=65, Exact=0.108, ±1=0.215, Bin=0.569
MST 8: N=130, Exact=0.185, ±1=0.431, Bin=0.269
MST 9: N=124, Exact=0.121, ±1=0.306, Bin=0.306
MST 10: N=110, Exact=0.027, ±1=0.118, Bin=0.282


### Casual Conversation v2

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set)
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 2\vgg16_lab_best.pth"
split_json = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 2\train_val_split.json"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv_mst3 = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\vgg16_model_2_predictions.csv"

metrics3, per_bin, df3 = evaluate_vgg16_regressor(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    output_range="sigmoid",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="label",
    person_id_column="subject_id",
    split_json=split_json,
    split_key="val_persons",      
    output_csv=output_csv,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating REGRESSOR model: G:\Thesis\CasualConversationv2_Dataset\Models\Model 2\vgg16_lab_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] Output range: sigmoid
[INFO] VGG feature dimension: 25088
[INFO] Loading split from: G:\Thesis\CasualConversationv2_Dataset\Models\Model 2\train_val_split.json
[INFO] Filtered from 184201 to 64876 images


Evaluating Regressor: 100%|██████████| 64876/64876 [13:56<00:00, 77.57it/s]


[INFO] Saved predictions → G:\Thesis\CasualConversationv2_Dataset\Models\Model 2\vgg16_predictions.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.3147
±1 bin tolerance:      0.7420
3-bin accuracy:        0.7159

=========== PER-BIN RESULTS (MST3) ===========
1: N=460, Exact=0.141, ±1=0.550, Bin=0.717
2: N=4923, Exact=0.236, ±1=0.550, Bin=0.550
3: N=12052, Exact=0.247, ±1=0.703, Bin=0.442
4: N=13924, Exact=0.271, ±1=0.782, Bin=0.688
5: N=20427, Exact=0.470, ±1=0.829, Bin=0.863
6: N=8614, Exact=0.244, ±1=0.756, Bin=0.899
7: N=2278, Exact=0.102, ±1=0.451, Bin=0.875
8: N=1536, Exact=0.171, ±1=0.537, Bin=0.401
9: N=614, Exact=0.393, ±1=0.803, Bin=0.803
10: N=48, Exact=0.146, ±1=0.458, Bin=0.792


### FACET

In [23]:
# Testing on FACET Dataset
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 2\vgg16_lab_best.pth"
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv=r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\vgg16_model_2_predictions.csv"

metrics, per_tone, df = evaluate_vgg16_regressor(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    output_range="sigmoid",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv,
    device="cuda",
    split_json=None
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating REGRESSOR model: G:\Thesis\CasualConversationv2_Dataset\Models\Model 2\vgg16_lab_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] Output range: sigmoid
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 2677 images


Evaluating Regressor: 100%|██████████| 2677/2677 [00:42<00:00, 62.98it/s]

[INFO] Saved predictions → G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\vgg16_model_2_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.2238
±1 tolerance accuracy: 0.5801
3-bin accuracy:        0.5738

=========== PER-TONE RESULTS ===========
MST 1: N=70, Exact=0.114, ±1=0.386, Bin=0.586
MST 2: N=559, Exact=0.238, ±1=0.510, Bin=0.510
MST 3: N=693, Exact=0.208, ±1=0.599, Bin=0.436
MST 4: N=485, Exact=0.229, ±1=0.674, Bin=0.666
MST 5: N=349, Exact=0.309, ±1=0.650, Bin=0.696
MST 6: N=288, Exact=0.233, ±1=0.625, Bin=0.778
MST 7: N=120, Exact=0.167, ±1=0.483, Bin=0.775
MST 8: N=67, Exact=0.075, ±1=0.373, Bin=0.239
MST 9: N=41, Exact=0.073, ±1=0.220, Bin=0.220
MST 10: N=5, Exact=0.000, ±1=0.000, Bin=0.000


### Model 3

### Monk Skin Tone Dataset

In [44]:
# Testing on Monk Skin Tone Dataset
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 3\vgg16_lab_best.pth"
csv_path = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\vgg16_model_3_predictions.csv"

metrics, per_tone, df = evaluate_vgg16_regressor(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    output_range="sigmoid",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv,
    device="cuda",
    split_json=None
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating REGRESSOR model: G:\Thesis\CasualConversationv2_Dataset\Models\Model 3\vgg16_lab_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] Output range: sigmoid
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 1388 images


Evaluating Regressor: 100%|██████████| 1388/1388 [00:37<00:00, 37.34it/s]

[INFO] Saved predictions → G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\vgg16_model_3_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.1960
±1 tolerance accuracy: 0.5814
3-bin accuracy:        0.6434

=========== PER-TONE RESULTS ===========
MST 1: N=174, Exact=0.006, ±1=0.362, Bin=0.730
MST 2: N=198, Exact=0.076, ±1=0.525, Bin=0.525
MST 3: N=86, Exact=0.081, ±1=0.360, Bin=0.093
MST 4: N=180, Exact=0.394, ±1=0.850, Bin=0.833
MST 5: N=167, Exact=0.222, ±1=0.689, Bin=0.778
MST 6: N=154, Exact=0.227, ±1=0.721, Bin=0.838
MST 7: N=65, Exact=0.338, ±1=0.785, Bin=0.769
MST 8: N=130, Exact=0.515, ±1=0.838, Bin=0.646
MST 9: N=124, Exact=0.137, ±1=0.516, Bin=0.516
MST 10: N=110, Exact=0.000, ±1=0.055, Bin=0.427


### Casual Conversation v2

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set)
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 3\vgg16_lab_best.pth"
split_json = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 3\train_val_split.json"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv_mst3 = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\vgg16_model_3_predictions.csv"

metrics3, per_bin, df3 = evaluate_vgg16_regressor(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    output_range="sigmoid",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    person_id_column="subject_id",
    split_json=split_json,
    split_key="val_persons",      
    output_csv=output_csv,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating REGRESSOR model: G:\Thesis\CasualConversationv2_Dataset\Models\Model 3\vgg16_lab_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] Output range: sigmoid
[INFO] VGG feature dimension: 25088
[INFO] Loading split from: G:\Thesis\CasualConversationv2_Dataset\Models\Model 3\train_val_split.json
[INFO] Filtered from 184201 to 64876 images


Evaluating Regressor: 100%|██████████| 64876/64876 [14:18<00:00, 75.60it/s]


[INFO] Saved predictions → G:\Thesis\CasualConversationv2_Dataset\Models\Model 3\vgg16_predictions.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.4009
±1 bin tolerance:      0.8491
3-bin accuracy:        0.7800

=========== PER-BIN RESULTS (MST3) ===========
1: N=460, Exact=0.002, ±1=0.489, Bin=0.904
2: N=4923, Exact=0.251, ±1=0.740, Bin=0.740
3: N=12052, Exact=0.451, ±1=0.901, Bin=0.552
4: N=13924, Exact=0.414, ±1=0.898, Bin=0.697
5: N=20427, Exact=0.495, ±1=0.900, Bin=0.912
6: N=8614, Exact=0.277, ±1=0.781, Bin=0.963
7: N=2278, Exact=0.133, ±1=0.485, Bin=0.917
8: N=1536, Exact=0.342, ±1=0.699, Bin=0.382
9: N=614, Exact=0.388, ±1=0.884, Bin=0.884
10: N=48, Exact=0.104, ±1=0.438, Bin=0.896


### FACET

In [24]:
# Testing on FACET Dataset
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 3\vgg16_lab_best.pth"
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv=r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\vgg16_model_3_predictions.csv"

metrics, per_tone, df = evaluate_vgg16_regressor(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    output_range="sigmoid",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv,
    device="cuda",
    split_json=None
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating REGRESSOR model: G:\Thesis\CasualConversationv2_Dataset\Models\Model 3\vgg16_lab_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] Output range: sigmoid
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 2677 images


Evaluating Regressor: 100%|██████████| 2677/2677 [00:41<00:00, 64.93it/s]

[INFO] Saved predictions → G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\vgg16_model_3_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.2652
±1 tolerance accuracy: 0.6746
3-bin accuracy:        0.6048

=========== PER-TONE RESULTS ===========
MST 1: N=70, Exact=0.000, ±1=0.129, Bin=0.414
MST 2: N=559, Exact=0.097, ±1=0.449, Bin=0.449
MST 3: N=693, Exact=0.300, ±1=0.749, Bin=0.349
MST 4: N=485, Exact=0.410, ±1=0.866, Bin=0.790
MST 5: N=349, Exact=0.375, ±1=0.831, Bin=0.854
MST 6: N=288, Exact=0.229, ±1=0.677, Bin=0.941
MST 7: N=120, Exact=0.275, ±1=0.600, Bin=0.900
MST 8: N=67, Exact=0.254, ±1=0.507, Bin=0.284
MST 9: N=41, Exact=0.049, ±1=0.390, Bin=0.390
MST 10: N=5, Exact=0.000, ±1=0.000, Bin=0.400


### Model 4

### Monk Skin Tone Dataset

In [47]:
# Testing on Monk Skin Tone Dataset
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 4\vgg16_lab_best.pth"
csv_path = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\vgg16_model_4_predictions.csv"

metrics, per_tone, df = evaluate_vgg16_regressor(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    output_range="sigmoid",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv,
    device="cuda",
    split_json=None
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating REGRESSOR model: G:\Thesis\CasualConversationv2_Dataset\Models\Model 4\vgg16_lab_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] Output range: sigmoid
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 1388 images


Evaluating Regressor: 100%|██████████| 1388/1388 [00:35<00:00, 38.69it/s]

[INFO] Saved predictions → G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\vgg16_model_4_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.1506
±1 tolerance accuracy: 0.4380
3-bin accuracy:        0.5014

=========== PER-TONE RESULTS ===========
MST 1: N=174, Exact=0.000, ±1=0.046, Bin=0.443
MST 2: N=198, Exact=0.010, ±1=0.318, Bin=0.318
MST 3: N=86, Exact=0.023, ±1=0.326, Bin=0.023
MST 4: N=180, Exact=0.411, ±1=0.822, Bin=0.922
MST 5: N=167, Exact=0.401, ±1=0.910, Bin=0.958
MST 6: N=154, Exact=0.312, ±1=0.773, Bin=0.922
MST 7: N=65, Exact=0.092, ±1=0.400, Bin=0.969
MST 8: N=130, Exact=0.077, ±1=0.423, Bin=0.077
MST 9: N=124, Exact=0.000, ±1=0.073, Bin=0.073
MST 10: N=110, Exact=0.000, ±1=0.000, Bin=0.036


### Casual Conversation v2

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set)
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 4\vgg16_lab_best.pth"
split_json = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 4\train_val_split.json"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv_mst4 = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\vgg16_model_4_predictions.csv"

metrics3, per_bin, df3 = evaluate_vgg16_regressor(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    output_range="sigmoid",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    person_id_column="subject_id",
    split_json=split_json,
    split_key="val_persons",      
    output_csv=output_csv,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating REGRESSOR model: G:\Thesis\CasualConversationv2_Dataset\Models\Model 5\vgg16_lab_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] Output range: sigmoid
[INFO] VGG feature dimension: 25088
[INFO] Loading split from: G:\Thesis\CasualConversationv2_Dataset\Models\Model 5\train_val_split.json
[INFO] Filtered from 184201 to 64876 images


Evaluating Regressor: 100%|██████████| 64876/64876 [14:17<00:00, 75.63it/s]


[INFO] Saved predictions → G:\Thesis\CasualConversationv2_Dataset\Models\Model 5\vgg16_predictions.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.3696
±1 bin tolerance:      0.8139
3-bin accuracy:        0.7515

=========== PER-BIN RESULTS (MST3) ===========
1: N=460, Exact=0.013, ±1=0.415, Bin=0.741
2: N=4923, Exact=0.126, ±1=0.550, Bin=0.550
3: N=12052, Exact=0.316, ±1=0.791, Bin=0.370
4: N=13924, Exact=0.387, ±1=0.891, Bin=0.790
5: N=20427, Exact=0.533, ±1=0.925, Bin=0.936
6: N=8614, Exact=0.312, ±1=0.805, Bin=0.962
7: N=2278, Exact=0.110, ±1=0.485, Bin=0.959
8: N=1536, Exact=0.143, ±1=0.423, Bin=0.155
9: N=614, Exact=0.197, ±1=0.612, Bin=0.612
10: N=48, Exact=0.021, ±1=0.271, Bin=0.771


### FACET

In [25]:
# Testing on FACET Dataset
lab_mean = np.array([27.715821371226003, 10.521480987873188, 8.514460146640673], dtype=np.float32)
lab_std  = np.array([24.70048803837073, 8.827357389195186, 8.660419910058293], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\Model 4\vgg16_lab_best.pth"
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv=r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\vgg16_model_4_predictions.csv"

metrics, per_tone, df = evaluate_vgg16_regressor(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    output_range="sigmoid",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    output_csv=output_csv,
    device="cuda",
    split_json=None
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact accuracy:        {metrics['accuracy_exact']:.4f}")
print(f"±1 tolerance accuracy: {metrics['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics['accuracy_3bins']:.4f}")

print("\n=========== PER-TONE RESULTS ===========")
for tone, stats in per_tone.items():
    print(f"MST {tone}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating REGRESSOR model: G:\Thesis\CasualConversationv2_Dataset\Models\Model 4\vgg16_lab_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] Output range: sigmoid
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 2677 images


Evaluating Regressor: 100%|██████████| 2677/2677 [00:41<00:00, 64.28it/s]


[INFO] Saved predictions → G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\vgg16_model_4_predictions.csv

=========== GLOBAL RESULTS ===========
Exact accuracy:        0.2010
±1 tolerance accuracy: 0.5614
3-bin accuracy:        0.5458

=========== PER-TONE RESULTS ===========
MST 1: N=70, Exact=0.000, ±1=0.057, Bin=0.214
MST 2: N=559, Exact=0.041, ±1=0.243, Bin=0.243
MST 3: N=693, Exact=0.152, ±1=0.479, Bin=0.172
MST 4: N=485, Exact=0.256, ±1=0.722, Bin=0.907
MST 5: N=349, Exact=0.387, ±1=0.914, Bin=0.960
MST 6: N=288, Exact=0.372, ±1=0.872, Bin=0.986
MST 7: N=120, Exact=0.267, ±1=0.650, Bin=0.942
MST 8: N=67, Exact=0.179, ±1=0.403, Bin=0.194
MST 9: N=41, Exact=0.000, ±1=0.146, Bin=0.146
MST 10: N=5, Exact=0.000, ±1=0.000, Bin=0.000


### Model Classification MST 10 (LAB)

### Monk Skin Tone Dataset

In [48]:
# Testing on Monk Skin Tone Dataset
lab_mean = np.array([33.618656158447266, 8.958210945129395, 8.925719261169434], dtype=np.float32)
lab_std = np.array([26.940208435058594, 8.05940055847168, 9.126977920532227], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\lab\vgg16_mst_best.pth" # Model trained on LAB inputs on balanced dataset & retrained the first layer
csv_path = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\mst10_predictions_lab.csv"

metrics3, per_bin, df3 = evaluate_vgg16_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    device="cuda",
    output_csv=output_csv,
    split_json=None
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating model: G:\Thesis\CasualConversationv2_Dataset\Models\lab\vgg16_mst_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 1388 images


Evaluating: 100%|██████████| 1388/1388 [00:34<00:00, 39.69it/s]

[INFO] Saved predictions: G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\mst10_predictions_lab.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.1852
±1 bin tolerance:      0.4748
3-bin accuracy:        0.5425

=========== PER-BIN RESULTS (MST3) ===========
1: N=174, Exact=0.006, ±1=0.207, Bin=0.511
2: N=198, Exact=0.157, ±1=0.278, Bin=0.278
3: N=86, Exact=0.047, ±1=0.326, Bin=0.093
4: N=180, Exact=0.394, ±1=0.706, Bin=0.783
5: N=167, Exact=0.263, ±1=0.904, Bin=0.922
6: N=154, Exact=0.162, ±1=0.532, Bin=0.682
7: N=65, Exact=0.200, ±1=0.600, Bin=0.800
8: N=130, Exact=0.385, ±1=0.523, Bin=0.415
9: N=124, Exact=0.121, ±1=0.508, Bin=0.508
10: N=110, Exact=0.027, ±1=0.091, Bin=0.291


### Casual Conversation v2

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set)
lab_mean = np.array([33.618656158447266, 8.958210945129395, 8.925719261169434], dtype=np.float32)
lab_std = np.array([26.940208435058594, 8.05940055847168, 9.126977920532227], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\lab\vgg16_mst_best.pth"
csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
split_json_mst3 = r"G:\Thesis\CasualConversationv2_Dataset\Models\lab\train_val_split.json"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\mst10_predictions_lab.csv"

metrics3, per_bin, df3 = evaluate_vgg16_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",       
    input_mode="lab",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="label",
    person_id_column="subject_id",
    split_json=split_json_mst3,
    split_key="val_persons",
    output_csv=output_csv,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating model: G:\Thesis\CasualConversationv2_Dataset\Models\lab\vgg16_mst_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] VGG feature dimension: 25088
[INFO] Loading split from: G:\Thesis\CasualConversationv2_Dataset\Models\lab\train_val_split.json
[INFO] Filtered from 184201 to 64635 images


Evaluating: 100%|██████████| 64635/64635 [24:57<00:00, 43.18it/s]  


[INFO] Saved predictions: G:\Thesis\CasualConversationv2_Dataset\Models\lab\mst_predictions_lab.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.6600
±1 bin tolerance:      0.8876
3-bin accuracy:        0.8735

=========== PER-BIN RESULTS (MST3) ===========
1: N=523, Exact=0.396, ±1=0.704, Bin=0.828
2: N=4928, Exact=0.682, ±1=0.846, Bin=0.846
3: N=11641, Exact=0.676, ±1=0.915, Bin=0.787
4: N=13610, Exact=0.658, ±1=0.870, Bin=0.831
5: N=21354, Exact=0.605, ±1=0.911, Bin=0.919
6: N=7685, Exact=0.776, ±1=0.896, Bin=0.964
7: N=2725, Exact=0.695, ±1=0.838, Bin=0.970
8: N=1475, Exact=0.702, ±1=0.791, Bin=0.758
9: N=579, Exact=0.661, ±1=0.820, Bin=0.820
10: N=115, Exact=0.617, ±1=0.635, Bin=0.965


### FACET

In [31]:
# Testing on FACET Dataset
lab_mean = np.array([33.618656158447266, 8.958210945129395, 8.925719261169434], dtype=np.float32)
lab_std = np.array([26.940208435058594, 8.05940055847168, 9.126977920532227], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\lab\vgg16_mst_best.pth" # Model trained on LAB inputs on balanced dataset & retrained the first layer
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\mst10_predictions_lab.csv"

metrics3, per_bin, df3 = evaluate_vgg16_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="lab",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    device="cuda",
    output_csv=output_csv,
    split_json=None
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating model: G:\Thesis\CasualConversationv2_Dataset\Models\lab\vgg16_mst_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: lab
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 2677 images


Evaluating: 100%|██████████| 2677/2677 [00:40<00:00, 65.50it/s]


[INFO] Saved predictions: G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\mst_10_predictions_lab.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.2107
±1 bin tolerance:      0.5637
3-bin accuracy:        0.5506

=========== PER-BIN RESULTS (MST3) ===========
1: N=70, Exact=0.014, ±1=0.214, Bin=0.400
2: N=559, Exact=0.186, ±1=0.372, Bin=0.372
3: N=693, Exact=0.134, ±1=0.560, Bin=0.274
4: N=485, Exact=0.264, ±1=0.654, Bin=0.790
5: N=349, Exact=0.309, ±1=0.805, Bin=0.840
6: N=288, Exact=0.358, ±1=0.684, Bin=0.833
7: N=120, Exact=0.125, ±1=0.575, Bin=0.842
8: N=67, Exact=0.149, ±1=0.299, Bin=0.239
9: N=41, Exact=0.049, ±1=0.293, Bin=0.293
10: N=5, Exact=0.000, ±1=0.400, Bin=0.600


### Model Classification MST 10 (RGB)

### Monk Skin Tone Dataset

In [49]:
# Testing on Monk Skin Tone Dataset
lab_mean = np.array([33.618656158447266, 8.958210945129395, 8.925719261169434], dtype=np.float32)
lab_std = np.array([26.940208435058594, 8.05940055847168, 9.126977920532227], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\Model 1\vgg16_mst_best.pth"
csv_path = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\mst10_predictions_rgb.csv"

metrics3, per_bin, df3 = evaluate_vgg16_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="rgb",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    device="cuda",
    output_csv=output_csv,
    split_json=None
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating model: G:\Thesis\CasualConversationv2_Dataset\Models\rgb\Model 1\vgg16_mst_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: rgb
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 1388 images


Evaluating: 100%|██████████| 1388/1388 [00:26<00:00, 51.84it/s]

[INFO] Saved predictions: G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\mst10_predictions_rgb.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.1844
±1 bin tolerance:      0.4863
3-bin accuracy:        0.5231

=========== PER-BIN RESULTS (MST3) ===========
1: N=174, Exact=0.000, ±1=0.201, Bin=0.500
2: N=198, Exact=0.162, ±1=0.303, Bin=0.303
3: N=86, Exact=0.058, ±1=0.209, Bin=0.070
4: N=180, Exact=0.300, ±1=0.850, Bin=0.928
5: N=167, Exact=0.449, ±1=0.850, Bin=0.868
6: N=154, Exact=0.188, ±1=0.786, Bin=0.844
7: N=65, Exact=0.354, ±1=0.600, Bin=0.938
8: N=130, Exact=0.246, ±1=0.638, Bin=0.285
9: N=124, Exact=0.048, ±1=0.153, Bin=0.153
10: N=110, Exact=0.000, ±1=0.045, Bin=0.127


### Casual Conversation v2

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set)
lab_mean = np.array([33.618656158447266, 8.958210945129395, 8.925719261169434], dtype=np.float32)
lab_std = np.array([26.940208435058594, 8.05940055847168, 9.126977920532227], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\Model 1\vgg16_mst_best.pth"
csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
split_json_mst3 = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\Model 1\train_val_split.json"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2" 
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\mst10_predictions_rgb.csv"

metrics3, per_bin, df3 = evaluate_vgg16_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",       # 3-class model
    input_mode="rgb",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    person_id_column="subject_id",
    split_json=split_json_mst3,
    split_key="val_persons",        # Use validation set
    output_csv=output_csv_mst3,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating model: G:\Thesis\CasualConversationv2_Dataset\Models\rgb\Model 1\vgg16_mst_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: rgb
[INFO] VGG feature dimension: 25088
[INFO] Loading split from: G:\Thesis\CasualConversationv2_Dataset\Models\rgb\Model 1\train_val_split.json
[INFO] Filtered from 184201 to 64635 images


Evaluating: 100%|██████████| 64635/64635 [07:47<00:00, 138.23it/s]


[INFO] Saved predictions: G:\Thesis\CasualConversationv2_Dataset\Models\Model 3\mst_predictions_mst3_rgb_val.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.7591
±1 bin tolerance:      0.9173
3-bin accuracy:        0.8975

=========== PER-BIN RESULTS (MST3) ===========
1: N=523, Exact=0.409, ±1=0.602, Bin=0.881
2: N=4928, Exact=0.678, ±1=0.859, Bin=0.859
3: N=11641, Exact=0.779, ±1=0.918, Bin=0.819
4: N=13610, Exact=0.730, ±1=0.943, Bin=0.870
5: N=21354, Exact=0.801, ±1=0.937, Bin=0.942
6: N=7685, Exact=0.768, ±1=0.926, Bin=0.960
7: N=2725, Exact=0.710, ±1=0.814, Bin=0.985
8: N=1475, Exact=0.707, ±1=0.855, Bin=0.776
9: N=579, Exact=0.689, ±1=0.896, Bin=0.896
10: N=115, Exact=0.913, ±1=0.922, Bin=0.957


### FACET

In [26]:
# Testing on FACET Dataset
lab_mean = np.array([33.618656158447266, 8.958210945129395, 8.925719261169434], dtype=np.float32)
lab_std = np.array([26.940208435058594, 8.05940055847168, 9.126977920532227], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\Model 1\vgg16_mst_best.pth"
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\mst10_predictions_rgb.csv"

metrics3, per_bin, df3 = evaluate_vgg16_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst10",
    input_mode="rgb",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    device="cuda",
    output_csv=output_csv,
    split_json=None
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating model: G:\Thesis\CasualConversationv2_Dataset\Models\rgb\Model 1\vgg16_mst_best.pth
[INFO] Label mode: mst10
[INFO] Input mode: rgb
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 2677 images


Evaluating: 100%|██████████| 2677/2677 [00:25<00:00, 103.29it/s]

[INFO] Saved predictions: G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\mst10_predictions_rgb.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.2275
±1 bin tolerance:      0.6156
3-bin accuracy:        0.5783

=========== PER-BIN RESULTS (MST3) ===========
1: N=70, Exact=0.000, ±1=0.129, Bin=0.457
2: N=559, Exact=0.211, ±1=0.449, Bin=0.449
3: N=693, Exact=0.202, ±1=0.645, Bin=0.345
4: N=485, Exact=0.282, ±1=0.790, Bin=0.734
5: N=349, Exact=0.347, ±1=0.819, Bin=0.837
6: N=288, Exact=0.229, ±1=0.688, Bin=0.875
7: N=120, Exact=0.117, ±1=0.342, Bin=0.850
8: N=67, Exact=0.149, ±1=0.328, Bin=0.194
9: N=41, Exact=0.073, ±1=0.220, Bin=0.220
10: N=5, Exact=0.000, ±1=0.400, Bin=0.400


### Model Classification MST 3 (RGB)

### Monk Skin Tone Dataset

In [50]:
# Testing on Monk Skin Tone Dataset
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\3mst\vgg16_mst_best.pth"
csv_path   = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\mst3_predictions_rgb.csv"

metrics3, per_bin, df3 = evaluate_vgg16_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst3",
    input_mode="rgb",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    device="cuda",
    output_csv=output_csv,
    split_json=None
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating model: G:\Thesis\CasualConversationv2_Dataset\Models\rgb\3mst\vgg16_mst_best.pth
[INFO] Label mode: mst3
[INFO] Input mode: rgb
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 1388 images


Evaluating: 100%|██████████| 1388/1388 [00:27<00:00, 51.30it/s]

[INFO] Saved predictions: G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\mst3_predictions_rgb.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.6383
±1 bin tolerance:      0.9791
3-bin accuracy:        0.6383

=========== PER-BIN RESULTS (MST3) ===========
Light (1-3): N=458, Exact=0.504, ±1=0.985, Bin=0.504
Mid (4-7): N=566, Exact=0.781, ±1=1.000, Bin=0.781
Dark (8-10): N=364, Exact=0.585, ±1=0.940, Bin=0.585


### Casual Conversation v2

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set)
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\3mst\vgg16_mst_best.pth"
split_json_mst3 = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\3mst\train_val_split.json"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv_mst3 = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\mst3_predictions_rgb.csv"

metrics3, per_bin, df3 = evaluate_vgg16_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst3",       # 3-class model
    input_mode="rgb",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    person_id_column="subject_id",
    split_json=split_json_mst3,
    split_key="val_persons",        # Use validation set
    output_csv=output_csv_mst3,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating model: G:\Thesis\CasualConversationv2_Dataset\Models\rgb\3mst\vgg16_mst_best.pth
[INFO] Label mode: mst3
[INFO] Input mode: rgb
[INFO] VGG feature dimension: 25088
[INFO] Loading split from: G:\Thesis\CasualConversationv2_Dataset\Models\rgb\3mst\train_val_split.json
[3571, 3639, 1388, 4923, 708, 5347, 3915, 101, 2875, 145, 3324, 2736, 3767, 1015, 922, 1423, 2760, 963, 5367, 4499, 743, 3926, 2372, 1392, 3716, 2969, 4406, 3947, 3073, 4858, 5020, 715, 4848, 897, 1684, 4247, 620, 65, 4830, 2451, 4313, 509, 2567, 3744, 2847, 1269, 4894, 4420, 2916, 5406, 843, 4607, 2502, 5013, 4183, 4962, 851, 1097, 1836, 1562, 3456, 4454, 858, 4255, 2485, 3814, 1578, 4122, 3705, 4414, 279, 4550, 70, 2419, 1580, 4399, 1389, 5366, 1059, 4775, 3719, 1838, 848, 8, 5125, 1917, 5138, 4901, 4061, 4644, 4254, 2215, 2610, 289, 4102, 5210, 401, 5305, 893, 1351, 3010, 4893, 991, 686, 3609, 196, 2933, 2514, 5543, 2928, 330, 311, 1069, 5392, 2459, 3759, 1558, 3979, 3242, 5168, 2582, 5190, 303, 5481, 4

Evaluating: 100%|██████████| 64635/64635 [16:38<00:00, 64.74it/s]


[INFO] Saved predictions: G:\Thesis\CasualConversationv2_Dataset\Models\rgb\3mst\mst_predictions_mst3_rgb_val.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.7530
±1 bin tolerance:      0.9992
3-bin accuracy:        0.7530

=========== PER-BIN RESULTS (MST3) ===========
Light (1-3): N=17092, Exact=0.604, ±1=0.999, Bin=0.604
Mid (4-7): N=45374, Exact=0.821, ±1=1.000, Bin=0.821
Dark (8-10): N=2169, Exact=0.518, ±1=0.984, Bin=0.518


### FACET

In [ ]:
# Testing on FACET Dataset
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\3mst\vgg16_mst_best.pth"
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\mst3_predictions_rgb.csv"

metrics3, per_bin, df3 = evaluate_vgg16_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst3",
    input_mode="rgb",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    device="cuda",
    output_csv=output_csv,
    split_json=None
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating model: G:\Thesis\CasualConversationv2_Dataset\Models\rgb\3mst\vgg16_mst_best.pth
[INFO] Label mode: mst3
[INFO] Input mode: rgb
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 2677 images


Evaluating: 100%|██████████| 2677/2677 [00:25<00:00, 106.59it/s]

[INFO] Saved predictions: G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\mst3_predictions_rgb.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.5917
±1 bin tolerance:      0.9970
3-bin accuracy:        0.5917

=========== PER-BIN RESULTS (MST3) ===========
Light (1-3): N=1322, Exact=0.484, ±1=0.996, Bin=0.484
Mid (4-7): N=1242, Exact=0.732, ±1=1.000, Bin=0.732
Dark (8-10): N=113, Exact=0.310, ±1=0.973, Bin=0.310


### Model Classification MST 3 (LAB)

### Monk Skin Tone Dataset

In [51]:
# Testing on Monk Skin Tone Dataset
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\lab\3mst\vgg16_mst_best.pth"
csv_path   = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations.csv"
image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\mst3_predictions_lab.csv"

metrics3, per_bin, df3 = evaluate_vgg16_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst3",
    input_mode="lab",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    device="cuda",
    output_csv=output_csv,
    split_json=None
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating model: G:\Thesis\CasualConversationv2_Dataset\Models\lab\3mst\vgg16_mst_best.pth
[INFO] Label mode: mst3
[INFO] Input mode: lab
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 1388 images


Evaluating: 100%|██████████| 1388/1388 [00:34<00:00, 39.77it/s]

[INFO] Saved predictions: G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\mst3_predictions_lab.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.5353
±1 bin tolerance:      0.9676
3-bin accuracy:        0.5353

=========== PER-BIN RESULTS (MST3) ===========
Light (1-3): N=458, Exact=0.367, ±1=0.945, Bin=0.367
Mid (4-7): N=566, Exact=0.751, ±1=1.000, Bin=0.751
Dark (8-10): N=364, Exact=0.412, ±1=0.945, Bin=0.412


### Casual Conversation v2

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set)
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\lab\3mst\vgg16_mst_best.pth"
split_json_mst3 = r"G:\Thesis\CasualConversationv2_Dataset\Models\lab\3mst\train_val_split.json"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv_mst3 = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\mst3_predictions_lab.csv"

metrics3, per_bin, df3 = evaluate_vgg16_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst3",       # 3-class model
    input_mode="lab",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    person_id_column="subject_id",
    split_json=split_json_mst3,
    split_key="val_persons",        # Use validation set
    output_csv=output_csv_mst3,
    device="cuda"
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating model: G:\Thesis\CasualConversationv2_Dataset\Models\lab\3mst\vgg16_mst_best.pth
[INFO] Label mode: mst3
[INFO] Input mode: lab
[INFO] VGG feature dimension: 25088
[INFO] Loading split from: G:\Thesis\CasualConversationv2_Dataset\Models\lab\3mst\train_val_split.json
[INFO] Filtered from 184201 to 64635 images


Evaluating: 100%|██████████| 64635/64635 [32:02<00:00, 33.61it/s]  


[INFO] Saved predictions: G:\Thesis\CasualConversationv2_Dataset\Models\lab\3mst\mst_predictions_mst3_rgb_val.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.7254
±1 bin tolerance:      0.9981
3-bin accuracy:        0.7254

=========== PER-BIN RESULTS (MST3) ===========
Light (1-3): N=17092, Exact=0.600, ±1=0.997, Bin=0.600
Mid (4-7): N=45374, Exact=0.789, ±1=1.000, Bin=0.789
Dark (8-10): N=2169, Exact=0.392, ±1=0.963, Bin=0.392


### FACET

In [ ]:
# Testing on FACET Dataset
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\lab\3mst\vgg16_mst_best.pth"
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\mst3_predictions_lab.csv"

metrics3, per_bin, df3 = evaluate_vgg16_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    label_mode="mst3",
    input_mode="lab",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="mst_label",
    device="cuda",
    output_csv=output_csv,
    split_json=None
)

print("\n=========== GLOBAL RESULTS (MST3) ===========")
print(f"Exact bin accuracy:    {metrics3['accuracy_exact']:.4f}")
print(f"±1 bin tolerance:      {metrics3['accuracy_pm1']:.4f}")
print(f"3-bin accuracy:        {metrics3['accuracy_3bins']:.4f}")

print("\n=========== PER-BIN RESULTS (MST3) ===========")
for bin_name, stats in per_bin.items():
    print(f"{bin_name}: N={stats['count']}, "
            f"Exact={stats['exact']:.3f}, "
            f"±1={stats['pm1']:.3f}, "
            f"Bin={stats['bin']:.3f}")

[INFO] Evaluating model: G:\Thesis\CasualConversationv2_Dataset\Models\lab\3mst\vgg16_mst_best.pth
[INFO] Label mode: mst3
[INFO] Input mode: lab
[INFO] VGG feature dimension: 25088
[INFO] No split JSON provided, using all 2677 images


Evaluating: 100%|██████████| 2677/2677 [00:40<00:00, 65.65it/s]

[INFO] Saved predictions: G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\mst3_predictions_lab.csv

=========== GLOBAL RESULTS (MST3) ===========
Exact bin accuracy:    0.5637
±1 bin tolerance:      0.9944
3-bin accuracy:        0.5637

=========== PER-BIN RESULTS (MST3) ===========
Light (1-3): N=1322, Exact=0.425, ±1=0.995, Bin=0.425
Mid (4-7): N=1242, Exact=0.741, ±1=1.000, Bin=0.741
Dark (8-10): N=113, Exact=0.239, ±1=0.920, Bin=0.239


## Ensamble Approach

In [13]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from PIL import Image
import json

import torch
import torch.nn as nn
from torchvision import transforms
from skimage.color import rgb2lab

###############################################################
# 1. Import the model class from your training script
###############################################################

from VGG16_reg_Cla_AnyLabel import (
    VGG16MSTClassifier,
    VGG16MSTRegressor
)


###############################################################
# 2. RGB Transform (same as training)
###############################################################

def rgb_eval_transform():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])


###############################################################
# 3. LAB Transform (same as training)
###############################################################

class EvalLABTransform:
    def __init__(self, lab_mean, lab_std):
        self.resize = transforms.Resize((224, 224))
        self.lab_mean = lab_mean.astype(np.float32)
        self.lab_std = lab_std.astype(np.float32)

    def __call__(self, img_pil):

        img = self.resize(img_pil)
        rgb = np.asarray(img).astype(np.float32) / 255.0

        lab = rgb2lab(rgb).astype(np.float32)
        lab_norm = (lab - self.lab_mean) / self.lab_std

        # Convert HWC → CHW
        return torch.from_numpy(lab_norm.transpose(2, 0, 1)).float()


###############################################################
# 4. HYBRID Transform (RGB + LAB concatenated)
###############################################################

class EvalHybridTransform:
    def __init__(self, lab_mean, lab_std):
        self.resize = transforms.Resize((224, 224))
        self.lab_mean = lab_mean.astype(np.float32)
        self.lab_std = lab_std.astype(np.float32)

        self.rgb_norm = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        )

    def __call__(self, img_pil):

        img = self.resize(img_pil)
        rgb_arr = np.asarray(img).astype(np.float32) / 255.0

        ###############
        # RGB BRANCH
        ###############
        rgb_tensor = torch.from_numpy(rgb_arr.transpose(2, 0, 1)).float()
        rgb_tensor = self.rgb_norm(rgb_tensor)

        ###############
        # LAB BRANCH
        ###############
        lab_arr = rgb2lab(rgb_arr).astype(np.float32)
        lab_norm = (lab_arr - self.lab_mean) / self.lab_std
        lab_tensor = torch.from_numpy(lab_norm.transpose(2, 0, 1)).float()

        ###############
        # CONCAT: [RGB||LAB] → 6 channels
        ###############
        return torch.cat([rgb_tensor, lab_tensor], dim=0)

def accuracy_top1(y_true, y_pred):
    return (y_true == y_pred).mean()


def confusion_metrics(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm

###############################################################
# 6. Main Evaluation Function (updated for mst3 and split JSON)
###############################################################
def evaluate_generic_classifier(
    model_path,
    csv_path,
    image_root,
    class_mapping,      # {0: "light", 1: "dark"}
    input_mode="rgb",
    lab_mean=None,
    lab_std=None,
    file_path_column="filename",
    label_column="label",
    person_id_column="person_id",
    split_json=None,
    split_key=None,
    output_csv="predictions.csv",
    device="cuda"
):
    device = torch.device(device)
    num_classes = len(class_mapping)

    model = VGG16MSTClassifier(
        input_mode=input_mode,
        num_classes=num_classes
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    df = pd.read_csv(csv_path)
    print(df)

    if split_json is not None:
        with open(split_json) as f:
            split_data = json.load(f)[split_key]
            person_ids = [int(x) for x in split_data]

        df = df[df[person_id_column].isin(person_ids)].reset_index(drop=True)

    print(df)

    transform = rgb_eval_transform() if input_mode == "rgb" else EvalLABTransform(lab_mean, lab_std)

    results = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_path = os.path.join(image_root, row[file_path_column])
        true_label = int(row[label_column]) - 1   # convert 1-based → 0-based

        img = Image.open(img_path).convert("RGB")
        img_tensor = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            pred_class = model(img_tensor).argmax(1).item()

        results.append({
            "image_path": img_path,
            "true_class": true_label,
            "pred_class": pred_class,
            "true_name": class_mapping[true_label],
            "pred_name": class_mapping[pred_class],
            "match": int(true_label == pred_class)
        })

    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv, index=False)

    print(out_df.keys)

    y_true = out_df["true_class"].values
    y_pred = out_df["pred_class"].values

    metrics = {
        "accuracy": accuracy_top1(y_true, y_pred),
        "confusion_matrix": confusion_metrics(y_true, y_pred, num_classes)
    }

    return metrics, out_df

def evaluate_light_dark_ensemble(
    stage1_model_path,   # light_dark vs medium
    stage2_model_path,   # light vs dark
    csv_path,
    image_root,
    true_label_mapping,  # e.g., {1: "light", 2: "medium", 3: "dark"}
    input_mode="rgb",
    lab_mean=None,
    lab_std=None,
    split_json=None,
    split_key=None,
    file_path_column="filename",
    label_column="label",
    person_id_column="person_id",
    output_csv="ensemble_predictions.csv",
    device="cuda"
):
    """
    Evaluates a two-stage ensemble classifier.
    
    Args:
        stage1_model_path: Path to stage 1 model (light_dark vs medium)
        stage2_model_path: Path to stage 2 model (light vs dark)
        csv_path: Path to CSV with columns [filename, label, person_id]
        image_root: Root directory containing images
        true_label_mapping: Dict mapping original CSV labels to class names
                           e.g., {1: "light", 2: "medium", 3: "dark"}
        input_mode: "rgb", "lab", or "hybrid"
        lab_mean: LAB normalization mean (required for lab/hybrid)
        lab_std: LAB normalization std (required for lab/hybrid)
        split_json: Optional path to JSON with train/val split
        split_key: Key in split_json to use ("train" or "val")
        file_path_column: Column name for image filename
        label_column: Column name for true label
        person_id_column: Column name for person ID
        output_csv: Path to save predictions
        device: Device to run inference on
    
    Returns:
        metrics: Dict with accuracy and confusion matrix
        out_df: DataFrame with predictions
    """
    device = torch.device(device)

    # ---------- STAGE 1 ----------
    stage1_mapping = {
        0: "light_dark",
        1: "medium"
    }

    stage2_mapping = {
        0: "light",
        1: "dark"
    }

    stage1 = VGG16MSTClassifier(input_mode=input_mode, num_classes=2)
    stage1.load_state_dict(torch.load(stage1_model_path, map_location=device))
    stage1.eval().to(device)

    stage2 = VGG16MSTClassifier(input_mode=input_mode, num_classes=2)
    stage2.load_state_dict(torch.load(stage2_model_path, map_location=device))
    stage2.eval().to(device)

    df = pd.read_csv(csv_path)

    if split_json:
        with open(split_json) as f:
            ids = json.load(f)[split_key]
        df = df[df[person_id_column].isin(ids)].reset_index(drop=True)

    # Create inverse mapping: class_name -> numeric label
    inverse_label_mapping = {v: k for k, v in true_label_mapping.items()}
    
    # Get unique class names for confusion matrix
    all_classes = sorted(set(true_label_mapping.values()))
    class_to_idx = {cls: idx for idx, cls in enumerate(all_classes)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}
    num_classes = len(all_classes)

    # Setup transforms
    if input_mode == "rgb":
        transform = rgb_eval_transform()
    elif input_mode == "lab":
        if lab_mean is None or lab_std is None:
            raise ValueError("lab_mean and lab_std required for LAB mode")
        transform = EvalLABTransform(lab_mean, lab_std)
    elif input_mode == "hybrid":
        if lab_mean is None or lab_std is None:
            raise ValueError("lab_mean and lab_std required for hybrid mode")
        transform = EvalHybridTransform(lab_mean, lab_std)
    else:
        raise ValueError(f"Unknown input_mode: {input_mode}")
    
    results = []

    print(f"[INFO] Evaluating ensemble on {len(df)} images...")
    print(df.head())

    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_path = os.path.join(image_root, row[file_path_column])
        true_label = int(row[label_column])   # original dataset label

        true_name = true_label_mapping[true_label]

        img = Image.open(img_path).convert("RGB")
        x = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            p1 = stage1(x).argmax(1).item()

        if p1 == 0:  # light/dark → pass to stage 2
            with torch.no_grad():
                p2 = stage2(x).argmax(1).item()
            final_pred = stage2_mapping[p2]
        else:
            final_pred = "medium"

        results.append({
            "image_path": img_path,
            "true_label": true_label,
            "true_name": true_name,
            "stage1_pred": stage1_mapping[p1],
            "final_pred": final_pred,
            "match": int(true_name == final_pred)
        })

    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv, index=False)

    # Convert to numeric indices for metrics calculation
    y_true_indices = np.array([class_to_idx[row["true_name"]] for _, row in out_df.iterrows()])
    y_pred_indices = np.array([class_to_idx[row["final_pred"]] for _, row in out_df.iterrows()])

    # Calculate metrics
    accuracy = accuracy_top1(y_true_indices, y_pred_indices)
    cm = confusion_metrics(y_true_indices, y_pred_indices, num_classes)

    metrics = {
        "accuracy": accuracy,
        "confusion_matrix": cm,
        "class_names": all_classes,
        "class_to_idx": class_to_idx
    }

    # Print summary
    print(f"\n{'='*60}")
    print(f"ENSEMBLE EVALUATION RESULTS")
    print(f"{'='*60}")
    print(f"Total images: {len(out_df)}")
    print(f"Overall Accuracy: {accuracy*100:.2f}%")
    print(f"\nConfusion Matrix:")
    print(f"{'':>10}", end="")
    for cls in all_classes:
        print(f"{cls:>10}", end="")
    print()
    for i, true_cls in enumerate(all_classes):
        print(f"{true_cls:>10}", end="")
        for j in range(num_classes):
            print(f"{cm[i, j]:>10}", end="")
        print()
    
    print(f"\nPer-class accuracy:")
    for i, cls in enumerate(all_classes):
        correct = cm[i, i]
        total = cm[i, :].sum()
        class_acc = 100.0 * correct / total if total > 0 else 0.0
        print(f"  {cls}: {class_acc:.2f}% ({correct}/{total})")
    
    print(f"{'='*60}\n")

    return metrics, out_df

def evaluate_hierarchical_mst_ensemble(
    stage1_model_path,   # light_dark vs medium
    stage2_model_path,   # light vs dark
    light_model_path,    # MST 1,2,3
    medium_model_path,   # MST 4,5,6,7
    dark_model_path,     # MST 8,9,10
    csv_path,
    image_root,
    input_mode="rgb",
    lab_mean=None,
    lab_std=None,
    split_json=None,
    split_key=None,
    file_path_column="filename",
    label_column="label",
    person_id_column="person_id",
    output_csv="hierarchical_ensemble_predictions.csv",
    device="cuda"
):
    """
    Evaluates a hierarchical ensemble classifier for MST (1-10) prediction.
    
    Pipeline:
    1. Stage 1: Classify as light_dark (0) or medium (1)
    2. Stage 2: If light_dark, classify as light (0) or dark (1)
    3. Stage 3: Based on coarse class, predict fine-grained MST:
       - light → MST 1,2,3
       - medium → MST 4,5,6,7
       - dark → MST 8,9,10
    
    Args:
        stage1_model_path: Path to stage 1 model (light_dark vs medium)
        stage2_model_path: Path to stage 2 model (light vs dark)
        light_model_path: Path to light MST model (3 classes: 1,2,3)
        medium_model_path: Path to medium MST model (4 classes: 4,5,6,7)
        dark_model_path: Path to dark MST model (3 classes: 8,9,10)
        csv_path: Path to CSV with columns [filename, label, person_id]
        image_root: Root directory containing images
        input_mode: "rgb", "lab", or "hybrid"
        lab_mean: LAB normalization mean (required for lab/hybrid)
        lab_std: LAB normalization std (required for lab/hybrid)
        split_json: Optional path to JSON with train/val split
        split_key: Key in split_json to use ("train" or "val")
        file_path_column: Column name for image filename
        label_column: Column name for true label
        person_id_column: Column name for person ID
        output_csv: Path to save predictions
        device: Device to run inference on
    
    Returns:
        metrics: Dict with accuracy, MAE, and confusion matrix
        out_df: DataFrame with predictions
    """
    device = torch.device(device)

    # ---------- LOAD ALL MODELS ----------
    print("[INFO] Loading ensemble models...")
    
    # Stage 1: light_dark vs medium
    stage1 = VGG16MSTClassifier(input_mode=input_mode, num_classes=2)
    stage1.load_state_dict(torch.load(stage1_model_path, map_location=device))
    stage1.eval().to(device)
    
    # Stage 2: light vs dark
    stage2 = VGG16MSTClassifier(input_mode=input_mode, num_classes=2)
    stage2.load_state_dict(torch.load(stage2_model_path, map_location=device))
    stage2.eval().to(device)
    
    # Stage 3a: Light MST (1,2,3)
    light_model = VGG16MSTClassifier(input_mode=input_mode, num_classes=3)
    light_model.load_state_dict(torch.load(light_model_path, map_location=device))
    light_model.eval().to(device)
    
    # Stage 3b: Medium MST (4,5,6,7)
    medium_model = VGG16MSTClassifier(input_mode=input_mode, num_classes=4)
    medium_model.load_state_dict(torch.load(medium_model_path, map_location=device))
    medium_model.eval().to(device)
    
    # Stage 3c: Dark MST (8,9,10)
    dark_model = VGG16MSTClassifier(input_mode=input_mode, num_classes=3)
    dark_model.load_state_dict(torch.load(dark_model_path, map_location=device))
    dark_model.eval().to(device)
    
    print("[INFO] All models loaded successfully.")

    # ---------- MST LABEL MAPPINGS ----------
    # Stage 3 model outputs (0-based indices) to actual MST labels
    light_mst_mapping = {0: 1, 1: 2, 2: 3}
    medium_mst_mapping = {0: 4, 1: 5, 2: 6, 3: 7}
    dark_mst_mapping = {0: 8, 1: 9, 2: 10}
    
    # Coarse category mappings
    stage1_mapping = {0: "light_dark", 1: "medium"}
    stage2_mapping = {0: "light", 1: "dark"}

    # ---------- LOAD AND FILTER DATA ----------
    df = pd.read_csv(csv_path)

    if split_json:
        with open(split_json) as f:
            ids = json.load(f)[split_key]
        df = df[df[person_id_column].isin(ids)].reset_index(drop=True)

    print(f"[INFO] Evaluating hierarchical ensemble on {len(df)} images...")
    print(df.head())

    # ---------- SETUP TRANSFORMS ----------
    if input_mode == "rgb":
        transform = rgb_eval_transform()
    elif input_mode == "lab":
        if lab_mean is None or lab_std is None:
            raise ValueError("lab_mean and lab_std required for LAB mode")
        transform = EvalLABTransform(lab_mean, lab_std)
    elif input_mode == "hybrid":
        if lab_mean is None or lab_std is None:
            raise ValueError("lab_mean and lab_std required for hybrid mode")
        transform = EvalHybridTransform(lab_mean, lab_std)
    else:
        raise ValueError(f"Unknown input_mode: {input_mode}")

    # ---------- RUN INFERENCE ----------
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Hierarchical Ensemble"):
        img_path = os.path.join(image_root, row[file_path_column])
        true_mst = int(row[label_column])  # True MST label (1-10)

        try:
            img = Image.open(img_path).convert("RGB")
            x = transform(img).unsqueeze(0).to(device)
        except Exception as e:
            print(f"[WARN] Failed to load {img_path}: {e}")
            continue

        with torch.no_grad():
            # Stage 1: light_dark vs medium
            stage1_pred = stage1(x).argmax(1).item()
            coarse_category = stage1_mapping[stage1_pred]
            
            # Stage 2 & 3: Determine fine-grained MST
            if stage1_pred == 0:  # light_dark
                # Stage 2: light vs dark
                stage2_pred = stage2(x).argmax(1).item()
                
                if stage2_pred == 0:  # light
                    # Stage 3a: Predict MST 1,2,3
                    light_pred = light_model(x).argmax(1).item()
                    final_mst = light_mst_mapping[light_pred]
                    coarse_category = "light"
                else:  # dark
                    # Stage 3c: Predict MST 8,9,10
                    dark_pred = dark_model(x).argmax(1).item()
                    final_mst = dark_mst_mapping[dark_pred]
                    coarse_category = "dark"
            else:  # medium
                # Stage 3b: Predict MST 4,5,6,7
                medium_pred = medium_model(x).argmax(1).item()
                final_mst = medium_mst_mapping[medium_pred]

        results.append({
            "image_path": img_path,
            "true_mst": true_mst,
            "pred_mst": final_mst,
            "coarse_category": coarse_category,
            "absolute_error": abs(true_mst - final_mst),
            "match": int(true_mst == final_mst)
        })

    # ---------- CREATE OUTPUT DATAFRAME ----------
    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv, index=False)
    print(f"[INFO] Predictions saved to {output_csv}")

    # ---------- CALCULATE METRICS ----------
    y_true = out_df["true_mst"].values
    y_pred = out_df["pred_mst"].values
    
    # Exact match accuracy
    accuracy = accuracy_top1(y_true, y_pred)
    
    # Mean Absolute Error
    mae = np.mean(np.abs(y_true - y_pred))
    
    # Confusion matrix (10 classes: MST 1-10)
    # Convert to 0-based indices for confusion matrix
    y_true_idx = y_true - 1
    y_pred_idx = y_pred - 1
    cm = confusion_metrics(y_true_idx, y_pred_idx, num_classes=10)
    
    # ±1 accuracy (prediction within 1 MST unit)
    within_1 = (np.abs(y_true - y_pred) <= 1).mean()
    
    metrics = {
        "accuracy": accuracy,
        "mae": mae,
        "within_1_accuracy": within_1,
        "confusion_matrix": cm,
        "mst_labels": list(range(1, 11))
    }

    # ---------- PRINT DETAILED RESULTS ----------
    print(f"\n{'='*70}")
    print(f"HIERARCHICAL MST ENSEMBLE EVALUATION RESULTS")
    print(f"{'='*70}")
    print(f"Total images: {len(out_df)}")
    print(f"Overall Accuracy: {accuracy*100:.2f}%")
    print(f"Within ±1 MST: {within_1*100:.2f}%")
    print(f"Mean Absolute Error: {mae:.3f}")
    
    print(f"\n{'='*70}")
    print(f"CONFUSION MATRIX (rows=true, cols=pred)")
    print(f"{'='*70}")
    print(f"{'MST':>5}", end="")
    for mst in range(1, 11):
        print(f"{mst:>5}", end="")
    print()
    print("-" * 70)
    
    for i in range(10):
        mst_label = i + 1
        print(f"{mst_label:>5}", end="")
        for j in range(10):
            print(f"{cm[i, j]:>5}", end="")
        print()
    
    print(f"\n{'='*70}")
    print(f"PER-CLASS METRICS")
    print(f"{'='*70}")
    print(f"{'MST':<6} {'Count':<8} {'Accuracy':<12} {'Avg Error':<12}")
    print("-" * 70)
    
    for i in range(10):
        mst_label = i + 1
        # Get samples for this MST
        mask = y_true == mst_label
        count = mask.sum()
        
        if count > 0:
            correct = cm[i, i]
            class_acc = 100.0 * correct / count
            
            # Average error for this class
            errors = np.abs(y_true[mask] - y_pred[mask])
            avg_error = errors.mean()
            
            print(f"{mst_label:<6} {count:<8} {class_acc:>6.2f}%{'':<5} {avg_error:>6.3f}")
        else:
            print(f"{mst_label:<6} {0:<8} {'N/A':<12} {'N/A':<12}")
    
    print(f"{'='*70}\n")

    return metrics, out_df

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set) - Prediction between (Light & Dark) vs Medium Skin Tones
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations_medium_other.csv"
model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\medium_other\vgg16_mst_best.pth"
split_json = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\medium_other\train_val_split.json"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\medium_other\mst_predictions_mst_medium_other_rgb_val.csv"

metrics, df = evaluate_generic_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    class_mapping={0: "light_dark", 1: "medium"},
    input_mode="rgb",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="label",
    person_id_column="person_id",
    split_json=split_json,
    split_key="val",
    device="cuda",
    output_csv=output_csv
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact bin accuracy: {metrics['accuracy']:.4f}")

In [ ]:
# Testing on Casual Conversation Dataset (Validation Set) - Prediction between Light vs Dark Skin Tones
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

csv_path   = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations_light_dark_only.csv"
model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\light_dark\vgg16_mst_best.pth"
split_json = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\light_dark\train_val_split.json"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\light_dark\mst_predictions_mst_light_dark_rgb_val.csv"

metrics_ld, df_ld = evaluate_generic_classifier(
    model_path=model_path,
    csv_path=csv_path,
    image_root=image_root,
    class_mapping={0: "light", 1: "dark"},
    input_mode="rgb",
    lab_mean=lab_mean,
    lab_std=lab_std,
    file_path_column="filename",
    label_column="label",
    person_id_column="person_id",
    split_json=split_json,
    split_key="val",
    device="cuda",
    output_csv=output_csv
)

print("\n=========== GLOBAL RESULTS ===========")
print(f"Exact bin accuracy: {metrics_ld['accuracy']:.4f}")

[INFO] VGG feature dimension: 25088
                                                filename  label  person_id
0      0007/0007_0007_portuguese_nonscripted_1_raw_fr...      1          7
1      0007/0007_0007_portuguese_nonscripted_1_raw_fr...      1          7
2      0007/0007_0007_portuguese_nonscripted_1_raw_fr...      1          7
3      0007/0007_0007_portuguese_nonscripted_1_raw_fr...      1          7
4      0007/0007_0007_portuguese_nonscripted_1_raw_fr...      1          7
...                                                  ...    ...        ...
54917  5556/5556_5556_english_nonscripted_5_raw_frame...      1       5556
54918  5556/5556_5556_english_scripted_0_raw_frame000...      1       5556
54919  5556/5556_5556_english_scripted_0_raw_frame000...      1       5556
54920  5556/5556_5556_english_scripted_0_raw_frame000...      1       5556
54921  5556/5556_5556_english_scripted_0_raw_frame000...      1       5556

[54922 rows x 3 columns]
                                      

100%|██████████| 19340/19340 [04:57<00:00, 65.11it/s] 


<bound method NDFrame.keys of                                               image_path  true_class  \
0      G:\Thesis\CasualConversationv2_Dataset\Segment...           0   
1      G:\Thesis\CasualConversationv2_Dataset\Segment...           0   
2      G:\Thesis\CasualConversationv2_Dataset\Segment...           0   
3      G:\Thesis\CasualConversationv2_Dataset\Segment...           0   
4      G:\Thesis\CasualConversationv2_Dataset\Segment...           0   
...                                                  ...         ...   
19335  G:\Thesis\CasualConversationv2_Dataset\Segment...           0   
19336  G:\Thesis\CasualConversationv2_Dataset\Segment...           0   
19337  G:\Thesis\CasualConversationv2_Dataset\Segment...           0   
19338  G:\Thesis\CasualConversationv2_Dataset\Segment...           0   
19339  G:\Thesis\CasualConversationv2_Dataset\Segment...           0   

       pred_class true_name pred_name  match  
0               0     light     light      1  
1          

In [54]:
# Testing on Monk Skin Tone Dataset - Ensamble of two above models 
# First predict if person is (Light/Dark) vs Medium -> If Light/Dark predict Light vs Dark
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

model_path_1 = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\medium_other\vgg16_mst_best.pth"
model_path_2 = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\light_dark\vgg16_mst_best.pth"

csv_path   = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations_3mst.csv"
image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\mst3_ensamble_rgb.csv"

metrics_ensamble, predictions_ensamble_df = evaluate_light_dark_ensemble(
    stage1_model_path=model_path_1,
    stage2_model_path=model_path_2,
    csv_path=csv_path,
    image_root=image_root,
    true_label_mapping={1: "light", 2: "medium", 3: "dark"},
    input_mode = 'rgb',
    split_json=None,
    split_key=None,
    device="cuda",
    output_csv=output_csv,
    label_column='mst_label'
)

print(f"Accuracy: {metrics_ensamble['accuracy']*100:.2f}%")

[INFO] VGG feature dimension: 25088
[INFO] VGG feature dimension: 25088
[INFO] Evaluating ensemble on 1388 images...
                                         filename  mst_label  subject_id
0  subject_18/PXL_20220922_183640936.PORTRAIT.jpg          1  subject_18
1  subject_18/PXL_20220922_183639710.PORTRAIT.jpg          1  subject_18
2  subject_18/PXL_20220922_183638393.PORTRAIT.jpg          1  subject_18
3  subject_18/PXL_20220922_183636542.PORTRAIT.jpg          1  subject_18
4  subject_18/PXL_20220922_183634690.PORTRAIT.jpg          1  subject_18


100%|██████████| 1388/1388 [00:28<00:00, 49.26it/s]



ENSEMBLE EVALUATION RESULTS
Total images: 1388
Overall Accuracy: 61.38%

Confusion Matrix:
                dark     light    medium
      dark       263        24        77
     light        19       233       206
    medium        83       127       356

Per-class accuracy:
  dark: 72.25% (263/364)
  light: 50.87% (233/458)
  medium: 62.90% (356/566)

Accuracy: 61.38%


In [ ]:
# Testing on Monk Skin Tone Dataset - Ensamble of two above models + Individual MST Label classifiers
# First predict if person is (Light/Dark) vs Medium -> If Light/Dark predict Light vs Dark -> Then for each group predict the exact MST Skin Tone Label
model_path_1 = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\medium_other\vgg16_mst_best.pth"
model_path_2 = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\light_dark\vgg16_mst_best.pth"
light_model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\light_1_2_3\vgg16_mst_best.pth"
medium_model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\medium_4_5_6_7\vgg16_mst_best.pth"
dark_model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\dark_8_9_10\vgg16_mst_best.pth"
csv_path   = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations_3mst.csv"
image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\mst_ensamble_eval.csv"

metrics, predictions = evaluate_hierarchical_mst_ensemble(
    stage1_model_path=model_path_1,
    stage2_model_path=model_path_2,
    light_model_path=light_model_path,
    medium_model_path=medium_model_path,
    dark_model_path=dark_model_path,
    csv_path=csv_path,
    image_root=image_root,
    input_mode="rgb",
    lab_mean=None,
    lab_std=None,
    split_json=None,
    split_key=None,
    output_csv=output_csv,
    device="cuda"
)

print(f"Exact Accuracy: {metrics['accuracy']*100:.2f}%")
print(f"Within ±1 MST: {metrics['within_1_accuracy']*100:.2f}%")
print(f"MAE: {metrics['mae']:.3f}")

[INFO] Loading ensemble models...
[INFO] VGG feature dimension: 25088
[INFO] VGG feature dimension: 25088
[INFO] VGG feature dimension: 25088
[INFO] VGG feature dimension: 25088
[INFO] VGG feature dimension: 25088
[INFO] All models loaded successfully.
[INFO] Evaluating hierarchical ensemble on 1388 images...
                                         filename  label   person_id
0  subject_18_PXL_20220922_183640936.PORTRAIT.jpg      1  subject_18
1  subject_18_PXL_20220922_183639710.PORTRAIT.jpg      1  subject_18
2  subject_18_PXL_20220922_183638393.PORTRAIT.jpg      1  subject_18
3  subject_18_PXL_20220922_183636542.PORTRAIT.jpg      1  subject_18
4  subject_18_PXL_20220922_183634690.PORTRAIT.jpg      1  subject_18


Hierarchical Ensemble: 100%|██████████| 1388/1388 [01:06<00:00, 21.02it/s]

[INFO] Predictions saved to G:\Thesis\CasualConversationv2_Dataset\Models\rgb\mst_ensamble_eval.csv

HIERARCHICAL MST ENSEMBLE EVALUATION RESULTS
Total images: 1388
Overall Accuracy: 4.76%
Within ±1 MST: 16.93%
Mean Absolute Error: 3.390

CONFUSION MATRIX (rows=true, cols=pred)
  MST    1    2    3    4    5    6    7    8    9   10
----------------------------------------------------------------------
    1    0   80  153   60  124   13    9    9   10    0
    2    0   46   81   90  182   29   55   64   19    0
    3    0    4   20    4   26   16   31  143  120    0
    4    0    0    0    0    0    0    0    0    0    0
    5    0    0    0    0    0    0    0    0    0    0
    6    0    0    0    0    0    0    0    0    0    0
    7    0    0    0    0    0    0    0    0    0    0
    8    0    0    0    0    0    0    0    0    0    0
    9    0    0    0    0    0    0    0    0    0    0
   10    0    0    0    0    0    0    0    0    0    0

PER-CLASS METRICS
MST    Count   

In [55]:
# Testing on Monk Skin Tone Dataset - Ensamble of two above models + Individual MST Label classifiers
# First predict if person is (Light/Dark) vs Medium -> If Light/Dark predict Light vs Dark -> Then for each group predict the exact MST Skin Tone Label
model_path_1 = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\medium_other\vgg16_mst_best.pth"
model_path_2 = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\light_dark\vgg16_mst_best.pth"
light_model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\light_1_2_3\vgg16_mst_best.pth"
medium_model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\medium_4_5_6_7\vgg16_mst_best.pth"
dark_model_path = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\dark_8_9_10\vgg16_mst_best.pth"

csv_path   = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\annotations_3mst.csv"
image_root = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE"
output_csv = r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\mst10_ensamble_rgb.csv"

metrics, predictions = evaluate_hierarchical_mst_ensemble(
    stage1_model_path=model_path_1,
    stage2_model_path=model_path_2,
    light_model_path=light_model_path,
    medium_model_path=medium_model_path,
    dark_model_path=dark_model_path,
    csv_path=csv_path,
    image_root=image_root,
    input_mode="rgb",
    lab_mean=None,
    lab_std=None,
    split_json=None,
    split_key=None,
    output_csv=output_csv,
    device="cuda",
    label_column="mst_label"
)

print(f"Exact Accuracy: {metrics['accuracy']*100:.2f}%")
print(f"Within ±1 MST: {metrics['within_1_accuracy']*100:.2f}%")
print(f"MAE: {metrics['mae']:.3f}")

[INFO] Loading ensemble models...
[INFO] VGG feature dimension: 25088
[INFO] VGG feature dimension: 25088
[INFO] VGG feature dimension: 25088
[INFO] VGG feature dimension: 25088
[INFO] VGG feature dimension: 25088
[INFO] All models loaded successfully.
[INFO] Evaluating hierarchical ensemble on 1388 images...
                                         filename  mst_label  subject_id
0  subject_18/PXL_20220922_183640936.PORTRAIT.jpg          1  subject_18
1  subject_18/PXL_20220922_183639710.PORTRAIT.jpg          1  subject_18
2  subject_18/PXL_20220922_183638393.PORTRAIT.jpg          1  subject_18
3  subject_18/PXL_20220922_183636542.PORTRAIT.jpg          1  subject_18
4  subject_18/PXL_20220922_183634690.PORTRAIT.jpg          1  subject_18


Hierarchical Ensemble: 100%|██████████| 1388/1388 [00:33<00:00, 41.09it/s]

[INFO] Predictions saved to G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\mst10_ensamble_rgb.csv

HIERARCHICAL MST ENSEMBLE EVALUATION RESULTS
Total images: 1388
Overall Accuracy: 4.76%
Within ±1 MST: 16.93%
Mean Absolute Error: 3.390

CONFUSION MATRIX (rows=true, cols=pred)
  MST    1    2    3    4    5    6    7    8    9   10
----------------------------------------------------------------------
    1    0   80  153   60  124   13    9    9   10    0
    2    0   46   81   90  182   29   55   64   19    0
    3    0    4   20    4   26   16   31  143  120    0
    4    0    0    0    0    0    0    0    0    0    0
    5    0    0    0    0    0    0    0    0    0    0
    6    0    0    0    0    0    0    0    0    0    0
    7    0    0    0    0    0    0    0    0    0    0
    8    0    0    0    0    0    0    0    0    0    0
    9    0    0    0    0    0    0    0    0    0    0
   10    0    0    0    0    0    0    0    0    0    0

PER-CLASS METRICS
MST    Count    Ac

In [15]:
# Testing on FACET Dataset - Ensamble of two above models 
# First predict if person is (Light/Dark) vs Medium -> If Light/Dark predict Light vs Dark
lab_mean = np.array([33.61865615, 8.95821094, 8.92571926], dtype=np.float32)
lab_std  = np.array([26.94020843, 8.05940056, 9.12697792], dtype=np.float32)

model_path_1 = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\medium_other\vgg16_mst_best.pth"
model_path_2 = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\light_dark\vgg16_mst_best.pth"
csv_path = r"G:\Thesis\FACET_Dataset\TESTING_Segmented_FACET_0.2_continuous\annotations_3mst.csv"
image_root = r"G:\Thesis\FACET_Dataset\TESTING_Segmented_FACET_0.2_continuous"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Models\rgb\facet_ensamble_eval.csv"

metrics_ensamble, predictions_ensamble_df = evaluate_light_dark_ensemble(
    stage1_model_path=model_path_1,
    stage2_model_path=model_path_2,
    csv_path=csv_path,
    image_root=image_root,
    true_label_mapping={1: "light", 2: "medium", 3: "dark"},
    input_mode = 'rgb',
    split_json=None,
    split_key=None,
    device="cuda",
    output_csv=output_csv
)

print(f"Accuracy: {metrics_ensamble['accuracy']*100:.2f}%")

[INFO] VGG feature dimension: 25088
[INFO] VGG feature dimension: 25088
[INFO] Evaluating ensemble on 2867 images...
         filename  label   person_id
0  sa_9697495.jpg      2  sa_9697495
1  sa_1932356.jpg      3  sa_1932356
2  sa_2886046.jpg      2  sa_2886046
3   sa_139877.jpg      2   sa_139877
4  sa_2588919.jpg      1  sa_2588919


100%|██████████| 2867/2867 [00:26<00:00, 106.27it/s]



ENSEMBLE EVALUATION RESULTS
Total images: 2867
Overall Accuracy: 58.95%

Confusion Matrix:
                dark     light    medium
      dark        44         6        66
     light        17       645       761
    medium        49       278      1001

Per-class accuracy:
  dark: 37.93% (44/116)
  light: 45.33% (645/1423)
  medium: 75.38% (1001/1328)

Accuracy: 58.95%


# Result Visualisation

In [ ]:
import pandas as pd
import numpy as np


# ==============================================================
# Bin mapping
# ==============================================================
BIN_LABELS = {
    0: "Light (1-3)",
    1: "Mid (4-7)",
    2: "Dark (8-10)",
}


def mst_to_bin(mst):
    if mst <= 3:
        return 0
    elif mst <= 7:
        return 1
    else:
        return 2


# ==============================================================
# Single CSV evaluation
# ==============================================================
def evaluate_mst_csv(csv_path):
    df = pd.read_csv(csv_path)

    # -----------------------------
    # Normalize column names
    # -----------------------------
    if "true_label" in df.columns:
        df["true_mst"] = df["true_label"]

    # -----------------------------
    # Determine bin source
    # -----------------------------
    if "true_mst" in df.columns and df["true_mst"].notna().any():
        df["true_mst"] = df["true_mst"].astype(int)
        df["bin_id"] = df["true_mst"].apply(mst_to_bin)
        # mode = "mst10"
    elif "true_bin" in df.columns:
        df["bin_id"] = df["true_bin"].astype(int)
        # mode = "mst3"
    else:
        raise ValueError("CSV must contain either true_mst or true_bin")

    # -----------------------------
    # Required metric columns
    # -----------------------------
    for col in ["match_exact", "match_pm1", "match_bin"]:
        if col not in df.columns:
            raise ValueError(f"Missing column: {col}")

    # -----------------------------
    # Global metrics
    # -----------------------------
    metrics = {
        "accuracy_exact": df["match_exact"].mean(),
        "accuracy_pm1": df["match_pm1"].mean(),
        "accuracy_3bins": df["match_bin"].mean(),
        "N": len(df),
        # "mode": mode,
    }

    # -----------------------------
    # Per-tone (only if MST10)
    # -----------------------------
    per_tone = {}
    if mode == "mst10":
        for tone in range(1, 11):
            sub = df[df["true_mst"] == tone]
            if len(sub) == 0:
                continue

            per_tone[tone] = {
                "N": len(sub),
                "exact": sub["match_exact"].mean(),
                "pm1": sub["match_pm1"].mean(),
                "bin": sub["match_bin"].mean(),
            }

    # -----------------------------
    # Per-bin (STRICT GROUPING)
    # -----------------------------
    per_bin = {}

    for bin_id in [0, 1, 2]:
        sub = df[df["bin_id"] == bin_id]
        if len(sub) == 0:
            continue

        per_bin[bin_id] = {
            "bin_label": BIN_LABELS[bin_id],
            "N": len(sub),
            "exact": sub["match_exact"].mean(),
            "pm1": sub["match_pm1"].mean(),
            "bin": sub["match_bin"].mean(),
        }

    return metrics, per_tone, per_bin, df


# ==============================================================
# Multi-model evaluation
# ==============================================================
def evaluate_multiple_mst_csvs(models):

    overall_rows = []
    per_tone_rows = []
    per_bin_rows = []

    for m in models:
        name = m["name"]
        csv_path = m["csv_path"]

        metrics, per_tone, per_bin, _ = evaluate_mst_csv(csv_path)

        # Overall
        overall_rows.append({
            "model": name,
            "N": metrics["N"],
            "accuracy_exact": metrics["accuracy_exact"],
            "accuracy_pm1": metrics["accuracy_pm1"],
            "accuracy_3bins": metrics["accuracy_3bins"],
            "mode": metrics["mode"],
        })

        # Per-tone
        for tone, stats in per_tone.items():
            per_tone_rows.append({
                "model": name,
                "mst_label": tone,
                "N": stats["N"],
                "exact": stats["exact"],
                "pm1": stats["pm1"],
                "bin": stats["bin"],
            })

        # Per-bin (EXACTLY 3 ROWS PER MODEL)
        for bin_id, stats in per_bin.items():
            per_bin_rows.append({
                "model": name,
                "bin_id": bin_id,
                "bin": stats["bin_label"],
                "N": stats["N"],
                "exact": stats["exact"],
                "pm1": stats["pm1"],
                "bin_acc": stats["bin"],
            })

    overall_df = pd.DataFrame(overall_rows).set_index("model")

    per_tone_df = (
        pd.DataFrame(per_tone_rows)
        .set_index(["mst_label", "model"])
        if per_tone_rows
        else pd.DataFrame(columns=["N", "exact", "pm1", "bin"])
    )

    per_bin_df = (
        pd.DataFrame(per_bin_rows)
        .set_index(["bin_id", "model"])
        .sort_index()
    )

    return overall_df, per_tone_df, per_bin_df


## Monk Skin Tone Metric Visualisation

In [74]:
models = [
    {
        "name": "MST - Skin Tone Classifier Library",
        "csv_path": r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv",
    },
    {
        "name": "MST - Random Forest",
        "csv_path": r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\rf_mst_predictions.csv",
    },
    {
        "name": "MST - DenseNet121",
        "csv_path": r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\densenet_predictions.csv",
    },
    {
        "name": "MST - VGG16 Model 1",
        "csv_path": r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\vgg16_model_1_predictions.csv",
    },
    {
        "name": "MST - VGG16 Model 2",
        "csv_path": r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\vgg16_model_2_predictions.csv",
    },
    {
        "name": "MST - VGG16 Model 3",
        "csv_path": r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\vgg16_model_3_predictions.csv",
    },
    {
        "name": "MST - VGG16 Model 4",
        "csv_path": r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\vgg16_model_4_predictions.csv",
    },
    {
        "name": "MST - VGG16 MST 10 LAB Model",
        "csv_path": r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\mst10_predictions_lab.csv",
    },
    {
        "name": "MST - VGG16 MST 10 RGB Model",
        "csv_path": r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\mst10_predictions_rgb.csv",
    },
    # {
    #     "name": "MST - VGG16 MST 3 LAB Model",
    #     "csv_path": r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\mst3_predictions_lab.csv",
    # },
    # {
    #     "name": "MST - VGG16 MST 3 RGB Model",
    #     "csv_path": r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\mst3_predictions_rgb.csv",
    # },
]

overall_df, per_tone_df, per_bin_df = evaluate_multiple_mst_csvs(models)

print("\n===== OVERALL =====")
display(overall_df.round(4))

print("\n===== PER TONE =====")
display(per_tone_df.round(4))

print("\n===== PER BIN =====")
display(per_bin_df.round(4))



===== OVERALL =====


,N,accuracy_exact,accuracy_pm1,accuracy_3bins,mode
model,,,,,
MST - Skin Tone Classifier Library,1388,0.1110,0.2961,0.4741,mst10
MST - Random Forest,1388,0.1318,0.3350,0.4099,mst10
MST - DenseNet121,1388,0.2104,0.5173,0.5641,mst10
MST - VGG16 Model 1,1388,0.2277,0.6203,0.6729,mst10
MST - VGG16 Model 2,1388,0.1859,0.4928,0.5389,mst10
MST - VGG16 Model 3,1388,0.1960,0.5814,0.6434,mst10
MST - VGG16 Model 4,1388,0.1506,0.4380,0.5014,mst10
MST - VGG16 MST 10 LAB Model,1388,0.1852,0.4748,0.5425,mst10
MST - VGG16 MST 10 RGB Model,1388,0.1844,0.4863,0.5231,mst10



===== PER TONE =====


,,N,exact,pm1,bin
mst_label,model,,,,
1,MST - Skin Tone Classifier Library,174,0.0000,0.0000,0.0000
2,MST - Skin Tone Classifier Library,198,0.0000,0.0000,0.0000
3,MST - Skin Tone Classifier Library,86,0.0000,0.0000,0.0000
4,MST - Skin Tone Classifier Library,180,0.0000,0.0000,0.6389
5,MST - Skin Tone Classifier Library,167,0.0000,0.1317,0.7844
...,...,...,...,...,...
6,MST - VGG16 MST 10 RGB Model,154,0.1883,0.7857,0.8442
7,MST - VGG16 MST 10 RGB Model,65,0.3538,0.6000,0.9385
8,MST - VGG16 MST 10 RGB Model,130,0.2462,0.6385,0.2846



===== PER BIN =====


bin    N   exact  \
bin_id model                                                                  
0      MST - DenseNet121                   Bin 1 – Light (1–3)  458  0.0808   
       MST - Random Forest                 Bin 1 – Light (1–3)  458  0.0000   
       MST - Skin Tone Classifier Library  Bin 1 – Light (1–3)  458  0.0000   
       MST - VGG16 MST 10 LAB Model        Bin 1 – Light (1–3)  458  0.0786   
       MST - VGG16 MST 10 RGB Model        Bin 1 – Light (1–3)  458  0.0808   
       MST - VGG16 Model 1                 Bin 1 – Light (1–3)  458  0.0546   
       MST - VGG16 Model 2                 Bin 1 – Light (1–3)  458  0.1485   
       MST - VGG16 Model 3                 Bin 1 – Light (1–3)  458  0.0502   
       MST - VGG16 Model 4                 Bin 1 – Light (1–3)  458  0.0087   
1      MST - DenseNet121                     Bin 2 – Mid (4–7)  566  0.2686   
       MST - Random Forest                   Bin 2 – Mid (4–7)  566  0.3233   
       MST - Skin Tone Classifier Library    Bin 2 – Mid (4–7)  566  0.0883   
       MST - VGG16 MST 10 LAB Model          Bin 2 – Mid (4–7)  566  0.2703   
       MST - VGG16 MST 10 RGB Model          Bin 2 – Mid (4–7)  566  0.3198   
       MST - VGG16 Model 1                   Bin 2 – Mid (4–7)  566  0.3799   
       MST - VGG16 Model 2                   Bin 2 – Mid (4–7)  566  0.2615   
       MST - VGG16 Model 3                   Bin 2 – Mid (4–7)  566  0.2915   
       MST - VGG16 Model 4                   Bin 2 – Mid (4–7)  566  0.3445   
2      MST - DenseNet121                   Bin 3 – Dark (8–10)  364  0.2830   
       MST - Random Forest                 Bin 3 – Dark (8–10)  364  0.0000   
       MST - Skin Tone Classifier Library  Bin 3 – Dark (8–10)  364  0.2857   
       MST - VGG16 MST 10 LAB Model        Bin 3 – Dark (8–10)  364  0.1868   
       MST - VGG16 MST 10 RGB Model        Bin 3 – Dark (8–10)  364  0.1044   
       MST - VGG16 Model 1                 Bin 3 – Dark (8–10)  364  0.2088   
       MST - VGG16 Model 2                 Bin 3 – Dark (8–10)  364  0.1154   
       MST - VGG16 Model 3                 Bin 3 – Dark (8–10)  364  0.2308   
       MST - VGG16 Model 4                 Bin 3 – Dark (8–10)  364  0.0275   

                                              pm1  bin_acc  
bin_id model                                                
0      MST - DenseNet121                   0.3581   0.3821  
       MST - Random Forest                 0.0349   0.0066  
       MST - Skin Tone Classifier Library  0.0000   0.0000  
       MST - VGG16 MST 10 LAB Model        0.2598   0.3319  
       MST - VGG16 MST 10 RGB Model        0.2467   0.3341  
       MST - VGG16 Model 1                 0.3952   0.4913  
       MST - VGG16 Model 2                 0.4105   0.4694  
       MST - VGG16 Model 3                 0.4323   0.5218  
       MST - VGG16 Model 4                 0.2162   0.3100  
1      MST - DenseNet121                   0.6254   0.6749  
       MST - Random Forest                 0.7845   1.0000  
       MST - Skin Tone Classifier Library  0.3145   0.6643  
       MST - VGG16 MST 10 LAB Model        0.7049   0.7986  
       MST - VGG16 MST 10 RGB Model        0.8039   0.8887  
       MST - VGG16 Model 1                 0.8587   0.9028  
       MST - VGG16 Model 2                 0.6873   0.7580  
       MST - VGG16 Model 3                 0.7597   0.8110  
       MST - VGG16 Model 4                 0.7862   0.9382  
2      MST - DenseNet121                   0.5495   0.6209  
       MST - Random Forest                 0.0137   0.0000  
       MST - Skin Tone Classifier Library  0.6401   0.7747  
       MST - VGG16 MST 10 LAB Model        0.3874   0.4093  
       MST - VGG16 MST 10 RGB Model        0.2940   0.1923  
       MST - VGG16 Model 1                 0.5330   0.5440  
       MST - VGG16 Model 2                 0.2940   0.2857  
       MST - VGG16 Model 3                 0.4918   0.5357  
       MST - VGG16 Model 4                 0.1758   0.0632

In [ ]:
models = [
    {
        "name": "MST - VGG16 MST 3 LAB Model",
        "csv_path": r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\mst3_predictions_lab.csv",
    },
    {
        "name": "MST - VGG16 MST 3 RGB Model",
        "csv_path": r"G:\Thesis\MonkSkinTone_Dataset\Segmented_MSTE\SkinToneEvaluation\mst3_predictions_rgb.csv",
    },
]

overall_df, per_tone_df, per_bin_df = evaluate_multiple_mst_csvs(models)

print("\n===== OVERALL =====")
display(overall_df.round(4))

print("\n===== PER BIN =====")
display(per_bin_df.round(4))



===== OVERALL =====


,N,accuracy_exact,accuracy_pm1,accuracy_3bins,mode
model,,,,,
MST - VGG16 MST 3 LAB Model,1388,0.5353,0.9676,0.5353,mst10
MST - VGG16 MST 3 RGB Model,1388,0.6383,0.9791,0.6383,mst10



===== PER BIN =====


bin    N   exact     pm1  \
bin_id model                                                                   
0      MST - VGG16 MST 3 LAB Model  Bin 1 – Light (1–3)  458  0.3668  0.9454   
       MST - VGG16 MST 3 RGB Model  Bin 1 – Light (1–3)  458  0.5044  0.9847   
1      MST - VGG16 MST 3 LAB Model    Bin 2 – Mid (4–7)  566  0.7509  1.0000   
       MST - VGG16 MST 3 RGB Model    Bin 2 – Mid (4–7)  566  0.7809  1.0000   
2      MST - VGG16 MST 3 LAB Model  Bin 3 – Dark (8–10)  364  0.4121  0.9451   
       MST - VGG16 MST 3 RGB Model  Bin 3 – Dark (8–10)  364  0.5852  0.9396   

                                    bin_acc  
bin_id model                                 
0      MST - VGG16 MST 3 LAB Model   0.3668  
       MST - VGG16 MST 3 RGB Model   0.5044  
1      MST - VGG16 MST 3 LAB Model   0.7509  
       MST - VGG16 MST 3 RGB Model   0.7809  
2      MST - VGG16 MST 3 LAB Model   0.4121  
       MST - VGG16 MST 3 RGB Model   0.5852

## CCv2 Metric Visualisation

In [ ]:
models = [
    # {
    #     "name": "CCv2 - Skin Tone Classifier Library",
    #     "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv",
    # },
    {
        "name": "CCv2 - Random Forest",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\rf_mst_predictions.csv",
    },
    {
        "name": "CCv2 - DenseNet121",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\densenet_predictions.csv",
    },
    {
        "name": "CCv2 - VGG16 Model 1",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\vgg16_model_1_predictions.csv",
    },
    {
        "name": "CCv2 - VGG16 Model 2",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\vgg16_model_2_predictions.csv",
    },
    {
        "name": "CCv2 - VGG16 Model 3",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\vgg16_model_3_predictions.csv",
    },
    {
        "name": "CCv2 - VGG16 Model 4",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\vgg16_model_4_predictions.csv",
    },
    {
        "name": "CCv2 - VGG16 MST 10 LAB Model",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\mst10_predictions_lab.csv",
    },
    {
        "name": "CCv2 - VGG16 MST 10 RGB Model",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\mst10_predictions_rgb.csv",
    }
]

overall_df, per_tone_df, per_bin_df = evaluate_multiple_mst_csvs(models)

print("\n===== OVERALL =====")
display(overall_df.round(4))

print("\n===== PER TONE =====")
display(per_tone_df.round(4))

print("\n===== PER BIN =====")
display(per_bin_df.round(4))


===== OVERALL =====


,N,accuracy_exact,accuracy_pm1,accuracy_3bins
model,,,,
CCv2 - Random Forest,64876,0.3463,0.7562,0.7001
CCv2 - DenseNet121,64876,0.3027,0.6915,0.6972
CCv2 - VGG16 Model 1,64876,0.3853,0.8445,0.7770
CCv2 - VGG16 Model 2,64876,0.3147,0.7420,0.7159
CCv2 - VGG16 Model 3,64876,0.4009,0.8491,0.7800
CCv2 - VGG16 Model 4,64876,0.3696,0.8139,0.7515
CCv2 - VGG16 MST 10 LAB Model,64635,0.6600,0.8876,0.8735
CCv2 - VGG16 MST 10 RGB Model,64635,0.7591,0.9173,0.8975



===== PER TONE =====


,,N,exact,pm1,bin
mst_label,model,,,,
1,CCv2 - Random Forest,460,0.0000,0.0000,0.2217
2,CCv2 - Random Forest,4923,0.0000,0.0504,0.0504
3,CCv2 - Random Forest,12052,0.0534,0.7373,0.0534
4,CCv2 - Random Forest,13924,0.6206,0.9784,0.9622
5,CCv2 - Random Forest,20427,0.6101,0.9892,0.9896
...,...,...,...,...,...
6,CCv2 - VGG16 MST 10 RGB Model,7685,0.7684,0.9264,0.9598
7,CCv2 - VGG16 MST 10 RGB Model,2725,0.7101,0.8143,0.9853
8,CCv2 - VGG16 MST 10 RGB Model,1475,0.7071,0.8549,0.7756


In [ ]:
models = [
    {
        "name": "CCv2 - VGG16 MST 3 LAB Model",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\mst3_predictions_lab.csv",
    },
    {
        "name": "CCv2 - VGG16 MST 3 RGB Model",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\SkinToneEvaluation\mst3_predictions_rgb.csv",
    },
]

overall_df, per_tone_df, per_bin_df = evaluate_multiple_mst_csvs(models)

print("\n===== OVERALL =====")
display(overall_df.round(4))

print("\n===== PER BIN =====")
display(per_bin_df.round(4))


## FACET Metric Visualisation

In [ ]:
models = [
    {
        "name": "FACET - Skin Tone Classifier Library",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\skin_tone_classifier_library_predictions.csv",
    },
    {
        "name": "FACET - Random Forest",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\rf_mst_predictions.csv",
    },
    {
        "name": "FACET - DenseNet121",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\densenet_predictions.csv",
    },
    {
        "name": "FACET - VGG16 Model 1",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\vgg16_model_1_predictions.csv",
    },
    {
        "name": "FACET - VGG16 Model 2",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\vgg16_model_2_predictions.csv",
    },
    {
        "name": "FACET - VGG16 Model 3",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\vgg16_model_3_predictions.csv",
    },
    {
        "name": "FACET - VGG16 Model 4",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\vgg16_model_4_predictions.csv",
    },
    {
        "name": "FACET - VGG16 MST 10 LAB Model",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\mst10_predictions_lab.csv",
    },
    {
        "name": "FACET - VGG16 MST 10 RGB Model",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\mst10_predictions_rgb.csv",
    }
]

overall_df, per_tone_df, per_bin_df = evaluate_multiple_mst_csvs(models)

print("\n===== OVERALL =====")
display(overall_df.round(4))

print("\n===== PER TONE =====")
display(per_tone_df.round(4))

print("\n===== PER BIN =====")
display(per_bin_df.round(4))



===== OVERALL =====


,N,accuracy_exact,accuracy_pm1,accuracy_3bins
model,,,,
FACET - Skin Tone Classifier Library,2677,0.0732,0.2394,0.4206
FACET - Random Forest,2677,0.1759,0.5499,0.4841
FACET - DenseNet121,2677,0.2099,0.5290,0.5760
FACET - VGG16 Model 1,2677,0.2682,0.6694,0.5999
FACET - VGG16 Model 2,2677,0.2238,0.5801,0.5738
FACET - VGG16 Model 3,2677,0.2652,0.6746,0.6048
FACET - VGG16 Model 4,2677,0.2010,0.5614,0.5458
FACET - VGG16 MST 10 LAB Model,2677,0.2107,0.5637,0.5506
FACET - VGG16 MST 10 RGB Model,2677,0.2275,0.6156,0.5783



===== PER TONE =====


,,N,exact,pm1,bin
mst_label,model,,,,
1,FACET - Skin Tone Classifier Library,70,0.0143,0.0286,0.0286
2,FACET - Skin Tone Classifier Library,559,0.0107,0.0161,0.0161
3,FACET - Skin Tone Classifier Library,693,0.0000,0.0072,0.0072
4,FACET - Skin Tone Classifier Library,485,0.0000,0.0433,0.8845
5,FACET - Skin Tone Classifier Library,349,0.0774,0.4900,0.8883
...,...,...,...,...,...
6,FACET - VGG16 MST 10 RGB Model,288,0.2292,0.6875,0.8750
7,FACET - VGG16 MST 10 RGB Model,120,0.1167,0.3417,0.8500
8,FACET - VGG16 MST 10 RGB Model,67,0.1493,0.3284,0.1940


In [ ]:
models = [
    {
        "name": "FACET - VGG16 MST 3 LAB Model",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\mst3_predictions_lab.csv",
    },
    {
        "name": "FACET - VGG16 MST 3 RGB Model",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\SkinToneEvaluation\mst3_predictions_rgb.csv",
    },
]

overall_df, per_tone_df, per_bin_df = evaluate_multiple_mst_csvs(models)

print("\n===== OVERALL =====")
display(overall_df.round(4))

print("\n===== PER BIN =====")
display(per_bin_df.round(4))


### Conclusion

For training all model were trained on a 65% 35% split of the Casual Conversation v2 Dataset given it one of the few decently sized datasets which has MST Skin Tone Labels. All confidence labels were used i.e (Low / Medium / High) & images were preprocessed to segment and extract the face using Mediapipes FaceMesh library as outlined in the DatasetPrep.ipynb.

Given the overall lack of high accuracy outlined in the above models and the lack of viable pre-trained models, it would be best to utilise the VGG16 3MST Classification model which achieved a 75% accuracy on the CCv2 Validation set whilst achieving a decent generalisation of 64% accuarcy on the monk skin tone dataset. The only other comparitavely performant model was the Ensamble (medium & other) -> (light & dark) model which achieved 75% & 97% accaurcy on the CCV2 dataset respectively and generalisation accuracy of 61.38% on the Monk Skin Tone dataset.

Attempts were made to train a more accurate exact MST skin tone model in line with the literature, however the lack of available data made this infeasable. 